# Painel de auditorias ANP — Microsoft Fabric

Este notebook reproduz, em um único notebook do Fabric, o pipeline do app Streamlit
deste repositório (`streamlit_app.py` + `pipeline.py` + `db.py`): ingestão de PDFs de
auditoria da ANP, classificação/extração via LLM (Claude ou Ollama) e gravação no
Supabase (Postgres).

## Como usar

1. **Anexe um Lakehouse** a este notebook (menu à esquerda → *Add lakehouse*). Os arquivos
   de entrada são lidos da área **Files** desse Lakehouse.
2. Envie para o Lakehouse (arraste os arquivos na área *Files* do Fabric):
   - `Files/anp/catalogo.csv` — o catálogo de tipos de documento do SEI (mesmo CSV que o
     app pede no `st.file_uploader`: colunas `nº processo sei`, `documento`,
     `tipo_documento`, `status`, ou os equivalentes em inglês).
   - `Files/anp/pdfs/` — PDFs individuais para upload direto (um por um, como no
     primeiro `file_uploader` do app).
   - `Files/anp/lotes/` — arquivos `.zip` para processamento em lote (Batch API da
     Anthropic, ou fila síncrona no caso do Ollama) — mesma estrutura esperada pelo app:
     uma pasta de 1º nível por processo dentro de cada zip.
3. Preencha a seção **Configuração** abaixo com as credenciais do Supabase e do
   provedor de LLM.
4. Rode as células até a seção **Seleção de documentos** — ali você escolhe, entre os
   arquivos encontrados no Lakehouse, quais processar nesta execução (equivalente a
   marcar arquivos no `file_uploader` do Streamlit).
5. Rode as células de processamento (upload direto e/ou lote) e depois as células de
   visualização dos dados gravados no Supabase.

O pipeline é **idempotente**: cada arquivo é identificado pelo hash SHA-256 do seu
conteúdo, então rodar o notebook de novo sobre os mesmos arquivos não duplica nada
(a não ser que você marque "forçar reprocessamento").

Toda a lógica de negócio abaixo é uma cópia fiel de `pipeline.py`, `db.py` e das
funções de orquestração de `streamlit_app.py` deste repositório — só a parte de
interface (Streamlit → seleção de arquivos do Lakehouse + `print`/`display`) foi
adaptada para rodar em notebook.


## 0. Instalar dependências

In [1]:
%pip install -q pdfplumber pypdfium2 anthropic supabase requests ipywidgets


Note: you may need to restart the kernel to use updated packages.


## 1. Configuração

Preencha as credenciais abaixo. **Recomendado**: use um Azure Key Vault conectado via
`notebookutils.credentials.getSecret` (defina `USE_KEY_VAULT = True` e `KEY_VAULT_URL`)
em vez de digitar segredos diretamente neste notebook — evita deixar chaves gravadas em
texto claro se o notebook for compartilhado/exportado.


In [ ]:
import os

# ================== SEGREDOS ==================
# Opção A (recomendado): Azure Key Vault via notebookutils — os "nomes de segredo no
# Key Vault" abaixo são só um exemplo, ajuste para os nomes reais dos seus segredos.
USE_KEY_VAULT = False
KEY_VAULT_URL = ""  # ex: "https://meu-keyvault.vault.azure.net/"

# Opção B: preencher direto aqui (cuidado ao compartilhar/exportar este notebook com
# valores preenchidos — prefira a Opção A em produção).



def get_secret(name, default=None):
    if USE_KEY_VAULT and KEY_VAULT_URL:
        try:
            import notebookutils
            valor = notebookutils.credentials.getSecret(KEY_VAULT_URL, name)
            if valor:
                return valor
        except Exception:
            pass
    if SECRETS_MANUAIS.get(name):
        return SECRETS_MANUAIS[name]
    return os.environ.get(name, default)


SUPABASE_URL = get_secret("SUPABASE_URL")
SUPABASE_KEY = get_secret("SUPABASE_KEY")

LLM_PROVIDER = (get_secret("LLM_PROVIDER", "claude") or "claude").strip().lower()

ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = get_secret("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")

OLLAMA_HOST = get_secret("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = get_secret("OLLAMA_MODEL", "qwen2.5:7b-instruct")
# Contexto menor por padrão para modelo local: em 16GB de memória unificada, um
# prompt gigante empurra o KV cache além do que sobra depois do peso do modelo +
# sistema operacional. Ajuste para cima se sua máquina/capacidade aguentar.
OLLAMA_MAX_CHARS = int(get_secret("OLLAMA_MAX_CHARS", "40000"))
OLLAMA_NUM_CTX = int(get_secret("OLLAMA_NUM_CTX", "16384"))
OLLAMA_NUM_PREDICT = int(get_secret("OLLAMA_NUM_PREDICT", "4096"))
# Modelos de "raciocínio" (qwen3, deepseek-r1 etc.) mandam a resposta inteira para o
# campo "thinking" do Ollama e deixam "response" vazio quando o pensamento não é
# desligado — "think: false" faz o modelo responder direto no campo certo.
OLLAMA_THINK = (get_secret("OLLAMA_THINK", "false") or "false").strip().lower() in ("true", "1", "yes")

CLAUDE_MAX_CHARS = int(get_secret("CLAUDE_MAX_CHARS", "80000"))

# Lotes com esse número de arquivos ou mais não são aplicados automaticamente
# pela seção de verificação de resultados — ficam pendentes até processamento manual.
BATCH_MAX_ARQUIVOS = int(get_secret("BATCH_MAX_ARQUIVOS", "10"))

CONFIDENCE_THRESHOLD = 0.7
MAX_CHARS = OLLAMA_MAX_CHARS if LLM_PROVIDER == "ollama" else CLAUDE_MAX_CHARS
CURRENT_MODEL_LABEL = f"ollama:{OLLAMA_MODEL}" if LLM_PROVIDER == "ollama" else f"claude:{ANTHROPIC_MODEL}"

# Limite maior que o padrão (8000/4096) só pro prompt de evidência de anexo — uma NC
# "guarda-chuva" (ex: "gestão de risco") pode puxar evidência de um anexo grande o
# bastante pra estourar o padrão numa chamada de UMA NC só.
ANEXO_CLAUDE_MAX_TOKENS = 16000
ANEXO_OLLAMA_NUM_PREDICT = 8192

print(f"Provedor de LLM: {CURRENT_MODEL_LABEL}")
if not (SUPABASE_URL and SUPABASE_KEY):
    print("⚠️  Faltam SUPABASE_URL / SUPABASE_KEY — preencha SECRETS_MANUAIS (ou o Key Vault) antes de processar.")
if LLM_PROVIDER == "claude" and not ANTHROPIC_API_KEY:
    print("⚠️  LLM_PROVIDER é 'claude', mas falta ANTHROPIC_API_KEY.")


## 2. Caminhos no Lakehouse

`/lakehouse/default/Files` é a área *Files* do Lakehouse anexado a este notebook —
ajuste `LAKEHOUSE_FILES_BASE` se você organizou os arquivos em outra subpasta.


In [3]:
LAKEHOUSE_FILES_BASE = "/lakehouse/default/Files/anp"
CATALOGO_CSV_PATH = f"{LAKEHOUSE_FILES_BASE}/catalogo.csv"
PDFS_DIR = f"{LAKEHOUSE_FILES_BASE}/pdfs"
LOTES_DIR = f"{LAKEHOUSE_FILES_BASE}/lotes"

for _p in (CATALOGO_CSV_PATH, PDFS_DIR, LOTES_DIR):
    print(("✅ " if os.path.exists(_p) else "❌ (não encontrado) ") + _p)


❌ (não encontrado) /lakehouse/default/Files/anp/catalogo.csv
❌ (não encontrado) /lakehouse/default/Files/anp/pdfs
❌ (não encontrado) /lakehouse/default/Files/anp/lotes


## 3. Pipeline — lógica pura

Cópia fiel de `pipeline.py` deste repositório: catálogo de tipos de documento do SEI,
extração de texto do PDF, regex de número de processo, prompts de classificação/
extração/re-casamento/evidência de anexo. Sem I/O de Supabase ou LLM.


In [ ]:
"""Lógica de extração de texto, catálogo de tipos de documento (CSV do SEI) e
prompt de classificação — compartilhada entre o app Streamlit (streamlit_app.py)
e scripts de teste/linha de comando.

O CSV do SEI (nº processo sei, documento, tipo_documento, data_captura, status)
é a fonte de verdade obrigatória do pipeline: só processamos documento que
esteja mapeado nele, e o número de processo usado pra vincular tudo vem dele
(não de regex no texto nem de suposição da IA). Usamos essa informação para
três coisas, ANTES de chamar a IA:

1. Descartar de cara arquivos com erro de captura, cujo tipo nunca carrega
   conteúdo de auditoria/não-conformidade (e-mail, recibo de protocolo, anexo
   solto, despacho de trâmite etc.), OU que simplesmente não estão mapeados
   no CSV — evita gastar tokens à toa e evita processar algo fora do escopo
   conhecido do processo.
2. Marcar com certeza os documentos que SÃO o Documento de Fiscalização (ou
   peça equivalente da família de fiscalização) como "auditoria_oficial",
   em vez de confiar na IA para adivinhar isso — essa era a principal fonte
   de classificação errada.
3. Fornecer o número de processo ANP autoritativo (coluna "nº processo sei")
   — qualquer número diferente encontrado no texto do documento é só
   registrado como divergência, nunca usado pra vínculo.
"""

import csv
import io
import re

import pdfplumber

# ---------------------------------------------------------------------------
# Número de processo ANP — extração determinística por regex
# ---------------------------------------------------------------------------
# Número de processo da ANP segue um formato fixo (ex: 48610.221768/2025-20).
# Extrair por regex direto do texto é mais confiável do que depender da IA
# "adivinhar" qual número do documento é o processo — evita que documentos do
# mesmo processo fiquem com o campo escrito de formas ligeiramente diferentes
# e, por isso, não se vinculem entre si.
#
# A classe [-‐‑‒–—−] cobre o hífen comum e os travessões "parecidos" que PDFs
# às vezes usam (en-dash, non-breaking hyphen etc.) — sem isso, dois
# documentos do mesmo processo podem extrair o número com traços diferentes
# e nunca dar match.
REGEX_PROCESSO_ANP = re.compile(r"\d{4,5}\.\d{6}/\d{4}[-‐‑‒–—−]\d{2}")


def extract_processo_regex(texto):
    # Só aceita um número que apareça perto de uma âncora que indica que é o
    # processo DESTE documento — rodapé padrão do SEI ("Processo nº ... SEI
    # nº ...") ou "Processo Administrativo ...". Sem isso, anexos técnicos que
    # citam processos de OUTRAS auditorias como precedente no meio do texto
    # (ex: "conforme processo 48610.222892/2024-21...") fariam essa regex
    # roubar um número errado e sobrescrever o que a IA extraiu corretamente.
    for busca in (texto, re.sub(r"\s+", " ", texto)):
        for m in REGEX_PROCESSO_ANP.finditer(busca):
            antes = busca[max(0, m.start() - 30):m.start()].lower()
            depois = busca[m.end():m.end() + 15].lower()
            if "administrativo" in antes or "sei" in depois:
                return normalize_processo(m.group(0))
    return None


def normalize_processo(numero):
    if not numero:
        return numero
    # unifica qualquer variação de travessão para o hífen comum
    for ch in "‐‑‒–—−":
        numero = numero.replace(ch, "-")
    return numero.strip()


# Pastas de processo em lotes ZIP costumam vir nomeadas como o próprio número
# do processo, só que com "_" no lugar da "/" (ex: "48610.229661_2024-49") —
# artefato comum de export do SEI, já que "/" não é válido em nome de pasta.
REGEX_PASTA_PROCESSO = re.compile(r"^(\d{4,5}\.\d{6})_(\d{4}-\d{2})$")


def extract_processo_from_foldername(nome_pasta):
    """Fallback de número de processo quando não há match do regex no texto.

    Usado no processamento em lote: se o PDF não tiver o rodapé padrão do SEI
    (ex: um Ofício mais curto), o nome da pasta ainda entrega o processo.
    """
    m = REGEX_PASTA_PROCESSO.match((nome_pasta or "").strip())
    if m:
        return f"{m.group(1)}/{m.group(2)}"
    return None


# ---------------------------------------------------------------------------
# Extração de texto
# ---------------------------------------------------------------------------
def extract_pages(file_bytes):
    """Texto de cada página do PDF, sem truncar — uma entrada por página.

    Usado tanto para montar o texto completo (`extract_text`, truncado pro
    limite de tokens do prompt) quanto para gerar a versão HTML navegável do
    documento e localizar em qual página um item foi citado."""
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        return [page.extract_text() or "" for page in pdf.pages]


def extract_text(file_bytes, max_chars):
    return "\n".join(extract_pages(file_bytes))[:max_chars]


def localizar_pagina_por_item(pages, numero_item):
    """Primeira página (1-based) que cita este numero_item como referência
    isolada (ex: "5.4") — usado pra ancorar o link de evidência na página
    certa do documento ORIGINAL (link_arquivo + fragmento "#page=N"; ver
    `_resolver_link_pagina` em streamlit_app.py). Heurística por regex (como
    `REGEX_SEI_REFERENCIADO`), não NLP:
    funciona porque o prompt já instrui a IA a extrair numero_item EXATAMENTE
    como aparece no texto do próprio documento. Devolve None se não achar
    (documento pode não repetir a numeração formal em lugar nenhum, ou o item
    veio de recasamento por assunto em vez do texto do próprio doc)."""
    if not numero_item:
        return None
    padrao = re.compile(r"(?<!\d)" + re.escape(numero_item) + r"(?!\d)")
    for i, texto_pagina in enumerate(pages, start=1):
        if padrao.search(texto_pagina or ""):
            return i
    return None


def _normaliza_busca(texto):
    return re.sub(r"\s+", " ", (texto or "")).strip()


def localizar_pagina_por_trecho(pages, trecho):
    """Primeira página (1-based) cujo texto contém este trecho verbatim —
    usado pra ancorar o link de evidência na página EXATA de onde ela foi
    extraída (mais preciso que `localizar_pagina_por_item`, que só acha a
    página onde o número do item é citado, não onde o texto da evidência em
    si está). Normaliza espaços/quebras de linha antes de comparar, já que
    pdfplumber pode quebrar linha de forma diferente do texto corrido que a
    IA viu. Trechos curtos demais dariam falso positivo, por isso o mínimo
    de caracteres. Devolve None se não achar (trecho pode ter sido levemente
    reescrito pela IA apesar da instrução, cruzar quebra de página, ou ser
    null quando a evidência não tem passagem direta de apoio no texto)."""
    trecho_norm = _normaliza_busca(trecho)
    if not trecho_norm or len(trecho_norm) < 8:
        return None
    for i, texto_pagina in enumerate(pages, start=1):
        if trecho_norm.lower() in _normaliza_busca(texto_pagina).lower():
            return i
    return None


# ---------------------------------------------------------------------------
# Catálogo de tipos de documento (CSV exportado do SEI)
# ---------------------------------------------------------------------------

# Tipos que são só metadado/roteamento administrativo do SEI, sem conteúdo de
# auditoria ou não-conformidade extraível — descartados sem chamar a IA.
# "Anexo" entra aqui porque hoje a própria IA já classifica esse tipo como
# "evidencia_anexa" e manda para revisão manual sem extrair nada automaticamente
# — ou seja, o texto era lido e mandado para o modelo sem nunca virar dado
# aproveitado. Melhor nem gastar o token.
TIPOS_DESCARTAR = {
    "e-mail",
    "recibo eletrônico de protocolo",
    "anexo",
    "publicação",
    "despacho de encaminhamento",
    "despacho para publicação",
    "despacho de proposta para deliberação da diretoria",
    "termo de encerramento de trâmite físico",
    "termo de arquivamento de processo eletrônico",
    "ficha para autuação de processo e distribuição",
    "certidão de distribuição",
    "certidão de redistribuição",
    "procuração",
    "boleto",
    "guia",
    "comprovante",
    "aviso",
    "pauta",
    "confirmação",
    "processo",
    "lista",
    "planilha",
    "ficha",
    "folha",
    "protocolo",
    "certidão",
    "certificado",
    "prontuário",
    "cota",
}

# Tipos que são o Documento de Fiscalização ou peças oficiais equivalentes da
# mesma família (ordem de serviço, plano e relatório de auditoria) — tratados
# como "auditoria_oficial" direto pelo tipo do SEI, sem depender da IA acertar
# a classificação.
TIPOS_PRIORITARIOS = {
    "documento de fiscalização (df)",
    "documento de fiscalização",
    "relatório de auditoria de segurança",
    "relatório de auditoria",
    "fiscalização: segurança operacional",
    "ordem de serviço de fiscalização",
    "plano de auditoria de segurança",
    "plano de auditoria",
}

# Subconjunto de TIPOS_PRIORITARIOS que é o Documento de Fiscalização em si
# (não a Ordem de Serviço, o Plano ou o Relatório, que também são
# "auditoria_oficial" mas não devem gerar não conformidade — só o DF é a
# fonte de não conformidades).
TIPOS_DF = {
    "documento de fiscalização (df)",
    "documento de fiscalização",
}

# Catálogo das 17 Práticas de Gestão do SGSO (Regulamento Técnico do Sistema
# de Gerenciamento da Segurança Operacional, ANP/2007), com seus subitens —
# usado no prompt de extração para preencher `norma_referencia` de cada não
# conformidade: número da prática ("N" ou "N.M") violada, extraído do próprio
# documento quando citado explicitamente, ou inferido pelo assunto usando este
# catálogo como referência quando o documento não citar a prática diretamente.
PRATICAS_GESTAO_SGSO = """1 - Cultura de Segurança, Compromisso e Responsabilidade Gerencial (1.2 Valores e Política de Segurança; 1.3 Estrutura Organizacional e Responsabilidade Gerencial; 1.4 Sistema de Comunicação; 1.5 Disponibilização e Planejamento de Recursos)
2 - Envolvimento do Pessoal (2.2 Participação do Pessoal)
3 - Qualificação, Treinamento e Desempenho do Pessoal (3.2 Estrutura Organizacional; 3.3 Treinamento)
4 - Ambiente de Trabalho e Fatores Humanos
5 - Seleção, Controle e Gerenciamento de Contratadas (5.2 Seleção e Avaliação de Contratadas; 5.3 Responsabilidades do Operador da Instalação; 5.4 Treinamento de Contratadas)
6 - Monitoramento e Melhoria Contínua do Desempenho (6.2 Indicadores e Metas de Desempenho de Segurança; 6.3 Monitoramento)
7 - Auditorias (7.2 Planejamento da Auditoria; 7.3 Execução da Auditoria; 7.4 Avaliação da Auditoria)
8 - Gestão da Informação e da Documentação (8.2 Responsabilidades na Gestão da Informação; 8.3 Acesso à Informação)
9 - Investigação de Incidentes (9.2 Procedimentos e Organização da Investigação; 9.3 Execução da Investigação; 9.4 Ações Corretivas; 9.5 Ações Preventivas)
10 - Projeto, Construção, Instalação e Desativação (10.2 Gestão e Organização; 10.3 Segurança nas Fases de Projeto, Construção, Instalação e Desativação)
11 - Elementos Críticos de Segurança Operacional (11.2 Identificação dos Elementos Críticos de Segurança Operacional; 11.3 Gerenciamento e Controle dos Elementos Críticos de Segurança Operacional)
12 - Identificação e Análise de Riscos (12.2 Tipos de Análise de Riscos; 12.3 Metodologia de Identificação e Análise de Riscos; 12.4 Execução da Análise de Riscos; 12.5 Elaboração do Relatório de Identificação e Análise de Riscos; 12.6 Resultados)
13 - Integridade Mecânica (13.2 Planejamento de Inspeção, Teste, Manutenção e Suprimento de Materiais; 13.3 Controle das Atividades; 13.4 Monitoramento e Avaliação de Resultados)
14 - Planejamento e Gerenciamento de Grandes Emergências (14.2 Planejamento de Situações de Emergência; 14.3 Resposta a Grandes Emergências; 14.4 Gestão dos Recursos de Resposta; 14.5 Sistema de Comunicação; 14.6 Exercícios Simulados; 14.7 Revisão do Plano de Emergência)
15 - Procedimentos Operacionais (15.2 Elaboração e Controle dos Procedimentos Operacionais; 15.3 Procedimentos para Partida e Desativação; 15.4 Operações Simultâneas)
16 - Gerenciamento de Mudanças (16.2 Tipos de Mudanças; 16.3 Procedimentos de Controle)
17 - Práticas de Trabalho Seguro e Procedimentos de Controle em Atividades Especiais (17.2 Permissão de Trabalho; 17.3 Monitoramento)"""


def _norm_tipo(tipo):
    return (tipo or "").strip().lower()


def is_tipo_df(tipo_sei):
    """True só para o Documento de Fiscalização em si — não pra Ordem de
    Serviço, Plano ou Relatório, que são "auditoria_oficial" mas não devem
    gerar não conformidade (só o DF é a fonte de não conformidades)."""
    return _norm_tipo(tipo_sei) in TIPOS_DF


def is_tipo_anexo(tipo_sei):
    """True só para o tipo "Anexo" — usado pra decidir se vale a pena checar,
    ANTES de descartar, se algum outro documento já citou este anexo
    especificamente como fonte de mais detalhe de evidência. Só se aplica a
    este tipo: E-mail, Recibo e os demais tipos de TIPOS_DESCARTAR nunca
    passam por essa checagem, mesmo que algum texto cite o número deles."""
    return _norm_tipo(tipo_sei) == "anexo"


# O CSV do SEI vem com cabeçalhos em português ("nº processo sei", "documento",
# "tipo_documento"), mas outras fontes de export (ex: planilha baixada do
# Excel) usam cabeçalhos em inglês ("process_number", "document_name",
# "document_type") para o mesmo conteúdo. Sem isso, o código procurava só o
# nome em português, não achava a coluna, e carregava um catálogo VAZIO sem
# avisar — a UI só dizia "catálogo não enviado" mesmo com o arquivo certo.
_ALIAS_COLUNAS = {
    "documento": ("documento", "document_name", "document", "nome_documento", "numero_documento"),
    "processo": (
        "nº processo sei", "n° processo sei", "no processo sei", "numero processo sei",
        "numero_processo_sei", "process_number", "processo", "numero_processo",
    ),
    "tipo_documento": ("tipo_documento", "tipo documento", "document_type"),
    "status": ("status", "situacao", "situação"),
}


def _norm_header(h):
    return re.sub(r"\s+", " ", (h or "").strip().lower())


def _resolver_colunas(fieldnames):
    """Acha, pra cada campo esperado, qual coluna do CSV corresponde — aceita
    tanto os cabeçalhos do SEI em português quanto variantes em inglês."""
    disponiveis = {_norm_header(fn): fn for fn in (fieldnames or [])}
    return {
        campo: next((disponiveis[_norm_header(c)] for c in candidatos if _norm_header(c) in disponiveis), None)
        for campo, candidatos in _ALIAS_COLUNAS.items()
    }


def load_catalog(csv_source):
    """Lê o CSV nº processo sei / documento / tipo_documento / status.

    Aceita um caminho de arquivo (str) ou um objeto file-like já aberto em
    modo texto (ex: io.StringIO, ou o retorno de um file_uploader do Streamlit).
    Reconhece tanto os cabeçalhos originais do SEI quanto variantes em inglês
    (ex: "process_number", "document_name", "document_type").

    Retorna dict {numero_documento(str): {"processo", "tipo_documento", "status"}}.
    """
    catalogo = {}

    def _parse(f):
        reader = csv.DictReader(f)
        colunas = _resolver_colunas(reader.fieldnames)
        if not colunas["documento"] or not colunas["processo"]:
            encontradas = ", ".join(reader.fieldnames or []) or "(nenhuma)"
            raise ValueError(
                "Não reconheci as colunas de documento/processo neste CSV. "
                f"Colunas encontradas: {encontradas}. "
                "Renomeie para 'documento' e 'nº processo sei' (ou 'document_name' / 'process_number')."
            )
        for row in reader:
            doc_id = (row.get(colunas["documento"]) or "").strip()
            if not doc_id:
                continue
            catalogo[doc_id] = {
                "processo": (row.get(colunas["processo"]) or "").strip(),
                "tipo_documento": (row.get(colunas["tipo_documento"]) or "").strip() if colunas["tipo_documento"] else "",
                "status": (row.get(colunas["status"]) or "").strip() if colunas["status"] else "",
            }

    if isinstance(csv_source, str):
        with open(csv_source, newline="", encoding="utf-8-sig") as f:
            _parse(f)
    else:
        _parse(csv_source)
    return catalogo


def doc_id_from_filename(filename):
    """'4516505.pdf' -> '4516505' (número do documento no SEI == nome do arquivo)."""
    base = filename.rsplit("/", 1)[-1].rsplit("\\", 1)[-1]
    return base.rsplit(".", 1)[0].strip()


def chave_ordenacao_arquivo(nome_arquivo):
    """Chave de ordenação numérica crescente pelo nome do arquivo — o nome é
    geralmente o número do documento no SEI, que cresce cronologicamente.
    Processar um lote nessa ordem ajuda um documento que cita/depende de
    outro do MESMO lote (ex: Parecer que cita o DF) resolver de primeira no
    processamento síncrono, sem precisar da fila de pendências. Cai pro nome
    como string se não for puramente numérico (defensivo — nem todo arquivo
    segue o padrão), sempre depois dos numéricos."""
    base = doc_id_from_filename(nome_arquivo)
    return (0, int(base)) if base.isdigit() else (1, base)


def classify_catalog_entry(entry):
    """Decide o que fazer com o arquivo ANTES de gastar tokens.

    Retorna "descartar" | "prioritario" | "processar".
    - "descartar": não chama a IA — arquivo é ruído administrativo, falhou a
      captura no SEI, OU não está mapeado no catálogo (o CSV é a fonte de
      verdade obrigatória: se o documento não está nele, não processamos).
    - "prioritario": chama a IA só para extrair campos, mas o tipo_documento
      final é forçado para "auditoria_oficial" (não depende da IA classificar).
    - "processar": segue o fluxo normal, IA decide o tipo_documento.
    """
    if not entry:
        return "descartar"
    if entry["status"] and entry["status"].strip().lower() != "sucesso":
        return "descartar"
    tipo = _norm_tipo(entry["tipo_documento"])
    if tipo in TIPOS_DESCARTAR:
        return "descartar"
    if tipo in TIPOS_PRIORITARIOS:
        return "prioritario"
    return "processar"


# ---------------------------------------------------------------------------
# Prompt de classificação/extração
# ---------------------------------------------------------------------------
PROMPT = """Você processa documentos de auditorias da ANP (regulação de petróleo e gás no Brasil).
Analise o texto abaixo e responda SOMENTE com JSON válido, sem markdown, sem texto fora do JSON.

Formato exato:
{{
  "tipo_documento": "auditoria_oficial|resposta_operadora|parecer_anp|evidencia_anexa|indeterminado",
  "confianca": 0.0,
  "numero_processo_anp": "string ou null",
  "numero_relatorio": "string ou null",
  "auditoria": null,
  "nao_conformidades": [],
  "respostas": []
}}
{dica_tipo_sei}
IDIOMA DOS CAMPOS: escreva os VALORES dos campos de texto livre listados abaixo (resumos,
descrições, categorias, ações, resultados, decisões etc.) EM INGLÊS, traduzindo do português do
documento original — os dados serão armazenados em inglês. Isso NÃO se aplica a "tipo_documento" nem
a "apresentado_por" (mantenha exatamente os valores literais especificados no formato). EXCEÇÕES que
devem permanecer exatamente como aparecem no documento original, SEM tradução: nomes próprios
(operadora, unidade_instalacao, auditor_responsavel, auditores_equipe); códigos, números e datas
(numero_processo_anp, numero_relatorio, codigo_auditoria_anp, ordem_servico, cnpj_operadora,
numero_item, norma_referencia, todas as datas); qualquer tag/código de equipamento dentro de
equipamento_evidencia (ex: mantenha "P-101", "PSV-204" intactos, mas pode traduzir o texto ao redor,
ex: "Bomba de incêndio P-101" -> "Fire pump P-101"); o número SEI entre parênteses citado dentro de
uma descricao de evidência (ex: mantenha "(4529951)" intacto mesmo traduzindo o texto ao redor, já
que esse número é extraído por regex depois); e o campo "trecho" de cada evidência, que deve ser
sempre uma cópia literal do texto original, no idioma original, nunca traduzido.

Preencha "auditoria" (objeto com codigo_auditoria_anp, ordem_servico, operadora, cnpj_operadora,
unidade_instalacao, tipo_auditoria, sumario_auditoria, acao_demandada, data_auditoria_inicio,
data_auditoria_fim, data_emissao_relatorio, auditor_responsavel, auditores_equipe, status_auditoria,
resultado_final, praticas_auditadas, documentos_solicitados, fonte_auditoria) SOMENTE se
tipo_documento for auditoria_oficial. O anexo técnico de uma auditoria também conta como
auditoria_oficial.

Preencha "nao_conformidades" (array de {{numero_item, descricao, categoria_nao_conformidade,
norma_referencia, classificacao_gravidade, acao_recomendada, prazo_correcao, severidade,
frequencia, elemento_critico, evidencias}}) SOMENTE se tipo_documento for auditoria_oficial E o
próprio documento for o Documento de Fiscalização (DF)
em si. Ordem de Serviço, Plano de Auditoria e Relatório de Auditoria também são auditoria_oficial,
mas NÃO são o DF — preencha "auditoria" normalmente para eles, e deixe "nao_conformidades" como
array vazio [], mesmo que o texto mencione problemas encontrados. Se houver dúvida sobre qual
documento é o DF, use a dica do sistema de origem (SEI) abaixo, quando presente.

Detalhe de cada campo novo de "auditoria":
- codigo_auditoria_anp: identificador interno da própria ANP para a auditoria, geralmente no
  formato "NNN_SSO_AAAA" (ex: "027_SSO_2025"), citado como "Auditoria 027_SSO_2025" no texto.
  Diferente do número do processo administrativo e do número do relatório/DF.
- ordem_servico: número da Ordem de Serviço, geralmente no cabeçalho do documento (ex: "OS SSO_002725").
- sumario_auditoria: resumo em até 40 palavras do que a auditoria constatou/tratou.
- acao_demandada: a medida cautelar ou ação exigida pela ANP como resultado da auditoria (ex:
  "Interdição da operação com hidrocarbonetos até saneamento das notificações"), em até 30 palavras.
- auditor_responsavel: o nome do auditor LÍDER/responsável pela auditoria.
- auditores_equipe: nomes dos DEMAIS integrantes da equipe de auditoria (além do auditor_responsavel),
  concatenados separados por "; " (ex: "João Silva; Maria Santos"). null se não houver outros nomes
  citados ou se só um auditor for mencionado.
- resultado_final: a conclusão/status final da auditoria SE o documento deixar isso claro (ex:
  "interditada", "liberada para operação"); use null se não houver essa informação.
- praticas_auditadas: as práticas/temas de segurança operacional que a auditoria teve como objetivo
  verificar (ex: "Gestão de mudanças", "Investigação de incidentes", "Integridade mecânica",
  "Gerenciamento de risco de processo"), concatenadas separadas por "; ". null se o documento não
  detalhar isso (ex: um Ofício curto, sem escopo descrito).
- documentos_solicitados: documentos que a ANP solicitou/exigiu que a operadora apresentasse ou
  comprovasse (ex: "Procedimento de partida e parada", "Registros de treinamento da equipe",
  "Certificados de inspeção de válvulas de alívio"), concatenados separados por "; ". null se o
  documento não listar isso.
- fonte_auditoria: o órgão que originou/conduziu a auditoria — normalmente "ANP", mas pode ser
  "Marinha" (Autoridade Marítima/Marinha do Brasil) ou outro órgão explicitamente citado como
  responsável pela fiscalização. Use "ANP" como padrão se o texto não indicar outro órgão claramente.

CATÁLOGO DE PRÁTICAS DE GESTÃO DO SGSO (Regulamento Técnico do Sistema de Gerenciamento da
Segurança Operacional, ANP) — use para preencher "norma_referencia" de cada não conformidade:
{praticas_gestao_sgso}

Detalhe de cada campo novo de "nao_conformidades":
- categoria_nao_conformidade: área temática do item (ex: "Gestão de risco", "Sistema de drenagem",
  "Integridade estrutural", "Comunicação", "Atmosfera explosiva"), em 2-4 palavras.
- norma_referencia: o número da Prática de Gestão do SGSO violada por este item, no formato "N" ou
  "N.M" (ex: "7", "13.2") — N de 1 a 17 conforme o CATÁLOGO acima, M o subitem quando o assunto
  corresponder especificamente a um subitem listado. PRIORIZE um número de prática/subitem citado
  explicitamente no próprio texto do documento (ex: "Prática de Gestão nº 7", "item 13.2 do SGSO",
  "descumprimento à prática de gestão 9") — copie esse número literalmente. Se o documento NÃO citar
  isso explicitamente, escolha, você mesmo, o número do CATÁLOGO acima que MELHOR corresponde ao
  ASSUNTO desta não conformidade (ex: um achado sobre plano de treinamento de emergência corresponde
  à prática 14; um achado sobre inspeção de válvulas corresponde à prática 13). Preencha sempre com o
  número mais próximo, mesmo por inferência — use null somente se o assunto não se encaixar em
  nenhuma das 17 práticas.
- acao_recomendada: a ação que a ANP exige da operadora para sanar especificamente este item,
  em até 20 palavras (geralmente o próprio texto da notificação, ex: "Refazer o estudo de
  dispersão de gases considerando condição de calmaria").
- severidade: classificação de severidade/impacto do risco associado a este item, SE o documento
  usar ou citar uma matriz/classificação de risco explícita (ex: "Catastrófica", "Crítica",
  "Moderada", "Leve"). Diferente de classificacao_gravidade (a classificação geral do achado) —
  severidade é especificamente o eixo de IMPACTO de uma matriz de risco. null se o documento não
  classificar isso.
- frequencia: classificação de frequência/probabilidade de ocorrência do risco associado a este
  item, SE o documento usar ou citar uma matriz/classificação de risco explícita (ex: "Frequente",
  "Provável", "Ocasional", "Remota", "Improvável"). null se o documento não classificar isso.
- elemento_critico: o Elemento Crítico de Segurança Operacional (ECS/barreira de segurança)
  relacionado a este item, quando o documento identificar (ex: "Sistema de detecção de gás",
  "Válvula de alívio de pressão", "Sistema de combate a incêndio", "Sistema de parada de emergência
  (ESD)"). null se não identificado.
- evidencias: ver seção "EVIDÊNCIAS" abaixo — evidências que fundamentam ESTA não conformidade
  específica (ex: registro fotográfico, laudo técnico, relatório citado como base do achado).

ATENÇÃO — numeração dos itens em Documentos de Fiscalização da ANP: é comum o documento ter DUAS
listas sobre os mesmos assuntos, com numerações diferentes:
  (a) uma seção descritiva/técnica (ex: "3-AUTO DE INTERDIÇÃO", itens "3.1", "3.2", "3.4"...),
      com o detalhamento de cada problema encontrado;
  (b) uma seção de NOTIFICAÇÃO formal (ex: "5-NOTIFICAÇÃO", itens "5.1", "5.4", "5.6"...), que
      instrui a operadora a comprovar/corrigir/apresentar evidências sobre cada assunto.
As respostas da operadora e os pareceres da ANP SEMPRE referenciam a numeração da seção (b), a de
NOTIFICAÇÃO. Portanto, o campo "numero_item" de cada não conformidade DEVE usar o número da seção
de notificação formal (ex: "5.4"), mesmo que você use o texto da seção descritiva (a) para preencher
"descricao" com mais detalhe. Nunca use a numeração da seção descritiva (ex: "3.1") como numero_item
se existir uma seção de notificação formal correspondente no mesmo documento.

Preencha "respostas" (array de {{numero_item, resultado_final, decisao_anp, resumo, data_resposta,
evidencias}}) SOMENTE se tipo_documento for resposta_operadora ou parecer_anp. numero_item deve
corresponder exatamente ao número usado pelo próprio documento ao citar a não
conformidade/condicionante (ex: "5.4"). resumo em até 25 palavras. data_resposta é a data de
emissão/assinatura DESTE documento de resposta/parecer (formato AAAA-MM-DD se possível; use null se
não encontrar) — não confundir com a data da não conformidade original.
{dica_ncs_existentes}

EVIDÊNCIAS — tanto em "nao_conformidades" quanto em "respostas", cada não conformidade normalmente
tem VÁRIAS evidências. "evidencias" é um array de {{descricao, trecho, apresentado_por,
sistema_evidencia, equipamento_evidencia, categoria_evidencia, descritivo_imagens}} — capture TODAS
as evidências distintas relacionadas àquele item específico (use array vazio [] se não houver nenhuma
pra esse item — nunca invente uma evidência que não esteja no texto):
- descricao: PRIORIZE o dado concreto e específico já descrito no próprio texto — medições, datas de
  ocorrências, quantidades, resultados de teste, contagens, fotos/registros mencionados — em vez de um
  resumo genérico. Exemplos de BOA descricao: "9 gas leaks detected since 2022, including 1 TIER 1
  event with CH4 and CO2 undetected by 2 detectors at the HH level", "Deluge test on 2/12/2025 in
  modules 3P+2PW showed drainage undersizing", "Foam tank at 68% of required volume (4,120 m³ of
  5,700 m³, capacity reduced by 32%)" (lembre-se: escreva descricao em inglês, ver instrução de
  IDIOMA DOS CAMPOS acima). Só descreva a evidência como citação de outro documento (ex: "Letter
  DPBR-2024-11589 (4529951)", "Anexo 03 (4529954)") quando o texto realmente aponta pra outro
  documento como fonte da prova, e não há detalhe observacional direto disponível no próprio texto.
  Quando o texto citar um documento (Carta, Ofício, Anexo, Parecer, laudo) como fonte de uma
  evidência, SEMPRE inclua o número SEI entre parênteses quando houver (ex: "(4529951)", "(SEI nº
  5093911)"), mantendo o número intacto — isso é usado para linkar a evidência ao arquivo certo
  depois.
- trecho: uma citação literal (8 a 30 palavras) copiada EXATAMENTE do texto do documento (mesmas
  palavras, mesma pontuação, SEMPRE no idioma original do documento — nunca traduza este campo, ao
  contrário de descricao) que contém o dado concreto usado em "descricao" — usado para localizar e
  recortar a página exata do PDF onde esta evidência aparece. Use null se não houver uma passagem
  contínua do texto que sustente diretamente essa evidência específica (ex: quando a evidência é só
  uma citação a outro documento, sem texto de apoio direto nesta página).
- apresentado_por: "operadora" se a evidência foi submetida pela empresa fiscalizada (ex: Cartas
  da operadora, anexos técnicos enviados por ela em resposta a uma notificação); "anp" se foi a
  própria ANP quem produziu/observou a evidência (ex: registro da fiscalização, laudo do auditor,
  parecer técnico anterior). Use null só se o texto não deixar claro quem apresentou.
- sistema_evidencia: o sistema operacional/de segurança ao qual esta evidência se refere (ex:
  "Sistema de drenagem", "Sistema de detecção e combate a incêndio", "Sistema elétrico", "Sistema
  de içamento"). null se não identificável no texto.
- equipamento_evidencia: o equipamento específico citado nesta evidência, se houver (ex: "Bomba de
  incêndio P-101", "Válvula PSV-204", "Detector de gás GD-12"). null se não identificável.
- categoria_evidencia: a natureza/tipo desta evidência (ex: "Registro fotográfico", "Laudo
  técnico", "Relatório de teste", "Documento", "Depoimento", "Medição"), em 1-3 palavras.
- descritivo_imagens: SE esta evidência envolver foto(s)/imagem(ns)/registro(s) visual(is) citados
  no texto, descreva o que a imagem mostra (ex: "Foto mostra corrosão acentuada na base do tanque,
  com placa de identificação visível"). null se a evidência não envolver imagem.
- Cuidado para não misturar evidências de itens diferentes: associe cada evidência apenas ao
  numero_item que o próprio texto vincula a ela, não à lista inteira de referências do documento.

Responda APENAS o JSON, começando com {{ e terminando com }}, sem nenhum texto antes ou depois.

TEXTO DO DOCUMENTO (pode estar truncado):
---
{texto}
---
"""


def build_prompt(texto, tipo_sei=None, nao_conformidades_existentes=None):
    dica = ""
    if tipo_sei:
        dica = (
            f'\nDica do sistema de origem (SEI): este documento foi catalogado lá como '
            f'"{tipo_sei}". Use isso como forte indício do tipo_documento correto, mas '
            f"confirme pelo conteúdo.\n"
        )

    dica_ncs = ""
    if nao_conformidades_existentes:
        linhas = "\n".join(
            f'- "{nc["numero_item"]}": {nc["descricao"]}'
            for nc in nao_conformidades_existentes if nc.get("numero_item")
        )
        if linhas:
            dica_ncs = (
                "NÃO CONFORMIDADES JÁ CADASTRADAS para este processo (extraídas do Documento de "
                "Fiscalização):\n"
                f"{linhas}\n"
                'Se este documento for resposta_operadora ou parecer_anp, o "numero_item" de cada item em '
                '"respostas" DEVE usar EXATAMENTE um dos numero_item da lista acima, escolhido pelo '
                "ASSUNTO tratado — mesmo que este documento organize sua própria análise com uma "
                'numeração diferente (ex: se este parecer usa "2.1", "2.2"... internamente mas isso não '
                "corresponde à numeração das notificações acima, IGNORE a numeração própria do documento "
                "e casa pelo assunto/conteúdo com o numero_item real da lista). Se o assunto de um item "
                "não corresponder a nenhuma não conformidade da lista, deixe numero_item como null em vez "
                "de inventar um número.\n"
            )

    return PROMPT.format(
        texto=texto,
        dica_tipo_sei=dica,
        dica_ncs_existentes=dica_ncs,
        praticas_gestao_sgso=PRATICAS_GESTAO_SGSO,
    )


# ---------------------------------------------------------------------------
# Prompt de "re-casamento" — usado quando uma resposta/parecer já processado
# não bateu com nenhuma não conformidade pelo numero_item (documento com
# numeração própria, sem relação com a do DF) e a IA precisa decidir, só pelo
# conteúdo já extraído, a qual não conformidade real ela se refere. Muito mais
# barato que reprocessar o documento inteiro — não usa o texto do PDF, só o
# resumo/decisão já extraídos na primeira passada.
# ---------------------------------------------------------------------------
PROMPT_REMATCH = """Você organiza auditorias de segurança operacional da ANP (petróleo e gás, Brasil).

Uma resposta/parecer já foi processado e extraído, mas o numero_item que o próprio documento usa
("{numero_item_original}") não corresponde a nenhuma não conformidade cadastrada para este processo —
provavelmente porque o documento usa uma numeração própria (ex: da sua seção de análise interna), sem
relação com a numeração das notificações do Documento de Fiscalização.

Resumo já extraído desta resposta: {resumo}
Decisão da ANP: {decisao_anp}

Não conformidades cadastradas para este processo:
{lista_ncs}

Qual numero_item da lista acima corresponde ao ASSUNTO tratado nesta resposta? Responda SOMENTE com
JSON, sem markdown, no formato exato {{"numero_item": "string ou null"}} — use null se o assunto não
corresponder claramente a nenhuma da lista.
"""


def build_rematch_prompt(resposta, nao_conformidades_existentes):
    lista = "\n".join(
        f'- "{nc["numero_item"]}": {nc["descricao"]}'
        for nc in nao_conformidades_existentes if nc.get("numero_item")
    )
    return PROMPT_REMATCH.format(
        numero_item_original=resposta.get("numero_item") or "(nenhum)",
        resumo=resposta.get("resumo") or "(sem resumo)",
        decisao_anp=resposta.get("decisao_anp") or "(sem decisão registrada)",
        lista_ncs=lista or "(nenhuma)",
    )


# ---------------------------------------------------------------------------
# Filtro local (Python, sem IA) de relevância de Anexo — decide CONTRA QUAIS
# não conformidades já cadastradas do processo um Anexo é candidato a ter
# relação, comparando vocabulário que já existe no banco pra cada NC com o
# texto do anexo. Só as NCs que baterem entram no prompt da IA; se nenhuma
# bater, a IA nem é chamada (decisão determinística, sem custo de token).
# ---------------------------------------------------------------------------
# Siglas/códigos técnicos (ex: "ESDV", "HAZOP", "QRA", "PSV-204", "CO2") —
# deliberadamente não usa lista de stopwords: só considera token específico
# em maiúsculas, nunca palavra genérica ("SISTEMA", "CONSIDERANDO" etc. não
# aparecem assim no texto corrido, então não geram falso positivo). Sem \b no
# final: extrai só o prefixo maiúsculo mesmo quando seguido de sufixo minúsculo
# (ex: "ESDVs" no plural, "HAZOPs") — o \b inicial já garante que o termo
# começa em borda de palavra.
REGEX_TERMO_TECNICO = re.compile(r"\b[A-ZÀ-Ú]{2,}[A-ZÀ-Ú0-9-]*")


def _termos_tecnicos(texto):
    return {t.upper() for t in REGEX_TERMO_TECNICO.findall(texto or "")}


def anexo_relevante_para_nc(texto_anexo, nc):
    """True se o anexo é candidato a ter relação com esta NC, por vocabulário
    já conhecido dela: frase de categoria_nao_conformidade ou elemento_critico
    aparecendo no texto do anexo, OU overlap de termos técnicos (siglas/
    códigos) entre a descricao da NC e o texto do anexo."""
    texto_lower = (texto_anexo or "").lower()
    for campo in (nc.get("categoria_nao_conformidade"), nc.get("elemento_critico")):
        if campo and campo.strip().lower() in texto_lower:
            return True
    termos_nc = _termos_tecnicos(nc.get("descricao") or "")
    if termos_nc and termos_nc & _termos_tecnicos(texto_anexo):
        return True
    return False


def filtrar_ncs_relevantes(texto_anexo, ncs):
    """Subconjunto de `ncs` (cada uma {numero_item, descricao,
    categoria_nao_conformidade, elemento_critico}, formato de
    `db.fetch_ncs_resumo`) que passa no filtro local de relevância — só essas
    entram no prompt da IA."""
    return [nc for nc in ncs if anexo_relevante_para_nc(texto_anexo, nc)]


# ---------------------------------------------------------------------------
# Prompt de "evidência de anexo" — usado pra ler um Anexo do processo contra
# UMA não conformidade candidata por vez (ver `filtrar_ncs_relevantes`), nunca
# a lista inteira numa chamada só. Um anexo grande com evidência pra várias
# NCs de uma vez gera uma resposta grande o bastante pra estourar max_tokens
# e voltar JSON truncado/inválido (já aconteceu — ver histórico de
# max_tokens em `_call_claude`); uma chamada pequena por NC candidata mantém
# cada resposta sempre dentro do limite — Anexo é raro no catálogo, e o
# filtro Python já reduz bastante quantas NCs viram chamada, então mais
# chamadas pequenas ainda sai barato.
# ---------------------------------------------------------------------------
PROMPT_ANEXO_EVIDENCIA = """Você organiza auditorias de segurança operacional da ANP (petróleo e gás, Brasil).

Este documento é um Anexo do processo, pré-selecionado (por vocabulário técnico em comum) como
CANDIDATO a sustentar a seguinte não conformidade já cadastrada:

Não conformidade {numero_item}: {descricao_nc}

Analise o texto do anexo e responda SOMENTE com JSON válido, sem markdown, no formato exato:
{{"evidencias": [{{"descricao": "string", "trecho": "string ou null",
"apresentado_por": "anp|operadora|null", "sistema_evidencia": "string ou null",
"equipamento_evidencia": "string ou null", "categoria_evidencia": "string ou null",
"descritivo_imagens": "string ou null"}}]}}

Confirme pelo conteúdo REAL do texto (não assuma só porque foi pré-selecionado) se este anexo
sustenta essa não conformidade específica. Se sim, extraia TODAS as evidências distintas
(observações, medições, datas, quantidades, resultados de teste, fotos mencionadas) — priorize
dado concreto sobre resumo genérico, mesmos critérios do prompt principal. Se o anexo não tiver
relação real com esta não conformidade, responda {{"evidencias": []}}. Nunca invente uma evidência
que não esteja no texto do anexo.

IDIOMA: escreva "descricao", "sistema_evidencia", "equipamento_evidencia" (exceto tags/códigos de
equipamento, ex: "P-101" — mantenha intactas), "categoria_evidencia" e "descritivo_imagens" EM
INGLÊS, traduzindo do português do anexo. "trecho" é a exceção: deve ser uma citação literal (8 a 30
palavras) copiada EXATAMENTE do texto do anexo, no idioma original, nunca traduzida — usada pra
recortar a página exata do PDF; use null se não houver passagem contínua de apoio direto. Mantenha
também intactos: qualquer número SEI entre parênteses citado dentro de descricao (ex: "(4529951)") e
o valor de apresentado_por (anp|operadora|null, sem tradução).

TEXTO DO ANEXO (pode estar truncado):
---
{texto}
---
"""


def build_anexo_evidencia_prompt(nc, texto_anexo):
    return PROMPT_ANEXO_EVIDENCIA.format(
        numero_item=nc.get("numero_item"),
        descricao_nc=nc.get("descricao") or "(sem descrição)",
        texto=texto_anexo,
    )


## 4. Acesso a dados — Supabase

Cópia fiel de `db.py` deste repositório: uma função por operação de tabela, mais os
helpers de upload para o Storage (evidências).


In [ ]:
"""Funções de acesso ao Supabase (Postgres) para as 4 tabelas do pipeline.

Trata o caso de upload em ordem aleatória: se um documento de resposta/parecer
chega antes da auditoria (ou de uma não conformidade específica) existir, o
resultado já extraído pela IA fica guardado em tb_arquivos_processados
(payload_json) e é retentado automaticamente sempre que uma auditoria nova é
gravada — sem precisar chamar a API de novo.
"""

import re
import uuid

from supabase import create_client


def get_client(url, key):
    return create_client(url, key)


def gen_id(prefix):
    return f"{prefix}-{uuid.uuid4().hex[:10]}"


def _norm(item):
    return (item or "").strip().lower()


def _norm_processo(numero):
    if not numero:
        return numero
    numero = numero.strip()
    for ch in "‐‑‒–—−":
        numero = numero.replace(ch, "-")
    return numero


# ---------------------------------------------------------------------------
# Controle de duplicidade / fila de pendências
# ---------------------------------------------------------------------------
def already_processed(sb, file_hash):
    res = sb.table("tb_arquivos_processados").select("*").eq("hash_arquivo", file_hash).execute()
    return res.data[0] if res.data else None


def upsert_arquivo_status(sb, file_hash, nome_arquivo, tipo_documento, confianca, status, motivo=None, payload_json=None,
                           numero_sei=None, numero_processo_anp=None, numero_processo_divergente=None, link_arquivo=None):
    sb.table("tb_arquivos_processados").upsert({
        "hash_arquivo": file_hash,
        "nome_arquivo": nome_arquivo,
        "tipo_documento": tipo_documento,
        "confianca": confianca,
        "status": status,
        "motivo": motivo,
        "payload_json": payload_json,
        "numero_sei": numero_sei,
        "numero_processo_anp": numero_processo_anp,
        "numero_processo_divergente": numero_processo_divergente,
        "link_arquivo": link_arquivo,
    }).execute()


def fetch_link_arquivo(sb, numero_sei):
    """Link do Storage (PDF original) de um arquivo já processado, buscado
    pelo número SEI (nome do arquivo sem extensão) — usado pra resolver o
    link de uma evidência pro arquivo que realmente tem o detalhe (não pro
    documento principal que só cita esse número)."""
    if not numero_sei:
        return None
    res = (
        sb.table("tb_arquivos_processados")
        .select("link_arquivo")
        .eq("numero_sei", numero_sei)
        .not_.is_("link_arquivo", "null")
        .limit(1)
        .execute()
    )
    return res.data[0]["link_arquivo"] if res.data else None


def fetch_pendentes(sb):
    res = (
        sb.table("tb_arquivos_processados")
        .select("*")
        .in_("status", ["pendente_vinculo", "parcial"])
        .execute()
    )
    return res.data


# ---------------------------------------------------------------------------
# tb_lotes_batch — processamento em lote via Batch API da Anthropic
# ---------------------------------------------------------------------------
def upsert_lote_batch(sb, id_lote, status, total_itens, itens_json, nome_arquivo_zip=None):
    payload = {
        "id_lote": id_lote,
        "status": status,
        "total_itens": total_itens,
        "itens_json": itens_json,
    }
    # só inclui se informado — evita apagar o nome já gravado quando esta
    # função é chamada de novo só pra atualizar status (process_pending_batches)
    if nome_arquivo_zip is not None:
        payload["nome_arquivo_zip"] = nome_arquivo_zip
    sb.table("tb_lotes_batch").upsert(payload).execute()


def fetch_lotes_batch(sb, status_excluir=None):
    res = sb.table("tb_lotes_batch").select("*").order("criado_em", desc=True).execute()
    dados = res.data
    if status_excluir:
        dados = [l for l in dados if l["status"] != status_excluir]
    return dados


# ---------------------------------------------------------------------------
# tb_auditorias
# ---------------------------------------------------------------------------
def find_auditoria(sb, numero_processo_anp=None, numero_relatorio=None):
    if numero_processo_anp:
        alvo = _norm_processo(numero_processo_anp)
        res = sb.table("tb_auditorias").select("*").execute()
        for row in res.data:
            if _norm_processo(row.get("numero_processo_anp")) == alvo:
                return row
    if numero_relatorio:
        res = sb.table("tb_auditorias").select("*").ilike("numero_relatorio", f"%{numero_relatorio}%").execute()
        if res.data:
            return res.data[0]
    return None


# Colunas de tb_auditorias que a IA pode preencher (exclui id_auditoria,
# arquivo_origem, criado_em, atualizado_em, que são geridas aqui). "dados"
# vem de um objeto JSON que a própria IA gerou e é espalhado (**dados) direto
# no payload — sem esse filtro, qualquer chave inesperada que o modelo
# inventasse (nome de campo diferente, campo extra) ia direto pro Postgrest e
# quebrava com "column not found", derrubando o processamento inteiro em vez
# de só marcar aquele arquivo como erro.
CAMPOS_AUDITORIA = {
    "numero_processo_anp", "numero_relatorio", "codigo_auditoria_anp", "ordem_servico",
    "operadora", "cnpj_operadora", "unidade_instalacao", "tipo_auditoria",
    "sumario_auditoria", "acao_demandada", "data_auditoria_inicio", "data_auditoria_fim",
    "data_emissao_relatorio", "auditor_responsavel", "auditores_equipe", "status_auditoria",
    "resultado_final", "praticas_auditadas", "documentos_solicitados", "fonte_auditoria",
}


def upsert_auditoria(sb, dados, arquivo_origem):
    dados = {k: v for k, v in dados.items() if k in CAMPOS_AUDITORIA}
    existente = find_auditoria(sb, dados.get("numero_processo_anp"), dados.get("numero_relatorio"))
    if existente:
        id_auditoria = existente["id_auditoria"]
        payload = {k: v for k, v in dados.items() if v is not None}
        if payload:
            sb.table("tb_auditorias").update(payload).eq("id_auditoria", id_auditoria).execute()
    else:
        id_auditoria = gen_id("AUD")
        payload = {"id_auditoria": id_auditoria, "arquivo_origem": arquivo_origem, **dados}
        sb.table("tb_auditorias").insert(payload).execute()
    return id_auditoria


def atualizar_arquivo_origem_df(sb, id_auditoria, arquivo_origem, score):
    """Só troca arquivo_origem (+ arquivo_origem_score) da auditoria se o novo
    score for MAIOR que o já salvo — garante que, quando há mais de um DF no
    mesmo processo, o mais informativo (mais não conformidades e mais campos
    de auditoria preenchidos) fique como referência, não importa a ordem de
    chegada dos arquivos."""
    res = sb.table("tb_auditorias").select("arquivo_origem_score").eq("id_auditoria", id_auditoria).execute()
    atual = (res.data[0].get("arquivo_origem_score") if res.data else 0) or 0
    if score > atual:
        sb.table("tb_auditorias").update({
            "arquivo_origem": arquivo_origem,
            "arquivo_origem_score": score,
        }).eq("id_auditoria", id_auditoria).execute()


# ---------------------------------------------------------------------------
# tb_nao_conformidades
# ---------------------------------------------------------------------------
def find_nc(sb, id_auditoria, numero_item):
    res = sb.table("tb_nao_conformidades").select("*").eq("id_auditoria", id_auditoria).execute()
    alvo = _norm(numero_item)
    for row in res.data:
        if _norm(row["numero_item"]) == alvo:
            return row
    return None


def fetch_ncs_resumo(sb, id_auditoria):
    """numero_item + descricao (+ categoria/elemento_critico, usados pelo
    filtro local de relevância de Anexo, ver `pipeline.filtrar_ncs_relevantes`)
    de cada NC da auditoria — usado pra dar contexto à IA (no prompt
    principal ou no re-casamento) sem precisar buscar a linha inteira."""
    res = (
        sb.table("tb_nao_conformidades")
        .select("numero_item,descricao,categoria_nao_conformidade,elemento_critico")
        .eq("id_auditoria", id_auditoria)
        .execute()
    )
    return [row for row in res.data if row.get("numero_item")]


def upsert_nc(sb, id_auditoria, nc, arquivo_origem, link_arquivo=None):
    existente = find_nc(sb, id_auditoria, nc.get("numero_item"))
    if existente:
        sb.table("tb_nao_conformidades").update({
            "descricao": nc.get("descricao") or existente["descricao"],
            "categoria_nao_conformidade": nc.get("categoria_nao_conformidade") or existente.get("categoria_nao_conformidade"),
            "norma_referencia": nc.get("norma_referencia") or existente.get("norma_referencia"),
            "classificacao_gravidade": nc.get("classificacao_gravidade") or existente.get("classificacao_gravidade"),
            "acao_recomendada": nc.get("acao_recomendada") or existente.get("acao_recomendada"),
            "prazo_correcao": nc.get("prazo_correcao") or existente.get("prazo_correcao"),
            "severidade": nc.get("severidade") or existente.get("severidade"),
            "frequencia": nc.get("frequencia") or existente.get("frequencia"),
            "elemento_critico": nc.get("elemento_critico") or existente.get("elemento_critico"),
        }).eq("id_nao_conformidade", existente["id_nao_conformidade"]).execute()
        id_nc = existente["id_nao_conformidade"]
    else:
        id_nc = gen_id("NC")
        sb.table("tb_nao_conformidades").insert({
            "id_nao_conformidade": id_nc,
            "id_auditoria": id_auditoria,
            "numero_item": nc.get("numero_item"),
            "descricao": nc.get("descricao"),
            "categoria_nao_conformidade": nc.get("categoria_nao_conformidade"),
            "norma_referencia": nc.get("norma_referencia"),
            "classificacao_gravidade": nc.get("classificacao_gravidade"),
            "acao_recomendada": nc.get("acao_recomendada"),
            "prazo_correcao": nc.get("prazo_correcao"),
            "severidade": nc.get("severidade"),
            "frequencia": nc.get("frequencia"),
            "elemento_critico": nc.get("elemento_critico"),
            "status_atual": "pending response",
            "arquivo_origem": arquivo_origem,
        }).execute()

    insert_evidencias(sb, nc.get("evidencias"), arquivo_origem, id_nao_conformidade=id_nc, link_arquivo=link_arquivo)
    return id_nc


# ---------------------------------------------------------------------------
# tb_respostas
# ---------------------------------------------------------------------------
def insert_resposta(sb, id_nc, id_auditoria, tipo_registro, r, arquivo_origem, link_arquivo=None):
    id_resposta = gen_id("RESP")
    sb.table("tb_respostas").insert({
        "id_resposta": id_resposta,
        "id_nao_conformidade": id_nc,
        "id_auditoria": id_auditoria,
        "tipo_registro": tipo_registro,
        "resultado_final": r.get("resultado_final"),
        "decisao_anp": r.get("decisao_anp"),
        "texto_resposta": r.get("resumo"),
        "data_resposta": r.get("data_resposta"),
        "arquivo_origem": arquivo_origem,
    }).execute()
    # Evidência é qualquer informação que sustente a não conformidade —
    # independente de quem apresentou (a origem já fica registrada por
    # evidência em "apresentado_por", anp ou operadora). Por isso grava pra
    # qualquer tipo_registro, não só resposta_operadora: um parecer_anp
    # normalmente cita evidência que a própria ANP levantou/observou.
    insert_evidencias(sb, r.get("evidencias"), arquivo_origem, id_resposta=id_resposta, link_arquivo=link_arquivo)
    if r.get("resultado_final"):
        sb.table("tb_nao_conformidades").update(
            {"status_atual": r.get("resultado_final")}
        ).eq("id_nao_conformidade", id_nc).execute()


# ---------------------------------------------------------------------------
# tb_evidencias — 1 linha por evidência, ligada a uma não conformidade OU a
# uma resposta (nunca as duas), com origem (ANP ou operadora). Uma NC ou
# resposta pode ter várias evidências, por isso é lista, não campo único.
# ---------------------------------------------------------------------------
APRESENTADO_POR_VALIDOS = {"anp", "operadora"}

# Acha o número SEI citado dentro da própria descrição da evidência — o texto
# já vem nesse formato na maioria dos casos ("Carta DPBR-2024-11589
# (4529951)", "Parecer nº 197 (4837346)", "(SEI nº 5093911)"). Em vez de
# depender da IA emitir um campo JSON separado pra isso, extrai por regex:
# mais simples e funciona mesmo em evidências já extraídas antes dessa
# mudança. 6-8 dígitos evita casar ano ("2019"), percentual etc.
REGEX_SEI_REFERENCIADO = re.compile(r"\((?:sei\s*n?º?\s*)?(\d{6,8})\)", re.IGNORECASE)


def _extrai_arquivo_referenciado(descricao):
    m = REGEX_SEI_REFERENCIADO.search(descricao or "")
    return m.group(1) if m else None


def _normaliza_apresentado_por(valor):
    valor = (valor or "").strip().lower()
    return valor if valor in APRESENTADO_POR_VALIDOS else None


def insert_evidencias(sb, evidencias, arquivo_origem, id_nao_conformidade=None, id_resposta=None, link_arquivo=None):
    """Grava as evidências de UMA não conformidade OU UMA resposta (nunca as
    duas — respeita a mesma regra da constraint em tb_evidencias).

    "evidencias" é o array bruto que a IA devolveu — trata formatos
    imperfeitos (string solta em vez de objeto, valor de apresentado_por fora
    do esperado) sem quebrar, e evita duplicar a mesma evidência se o mesmo
    item for reprocessado ou citado de novo por outro documento. Se a
    evidência já existe (mesma descrição) mas o link salvo é de um formato
    antigo (HTML, de antes de ancorar no PDF original), atualiza só o link em
    vez de pular — sem isso, reprocessar um documento nunca corrigia o link
    de evidências cuja descrição a IA volta a extrair igual.

    Prioridade do link de cada evidência:
    1. `ev["_link_pagina"]` — o link do documento original já ancorado pelo
       chamador na página certa (ver `_resolver_link_pagina`/
       `_resolver_paginas_evidencias` em streamlit_app.py: `link_arquivo` +
       fragmento "#page=N"), localizada pelo `trecho` (ou, na falta dele,
       pelo numero_item) desta evidência no documento atual — chave
       transiente em memória, nunca gravada no banco.
    2. Se não achou página própria, tenta achar (via regex, ver
       `_extrai_arquivo_referenciado`) um número SEI citado na própria
       descrição — se achar E já existir o link daquele arquivo em
       `tb_arquivos_processados` (`fetch_link_arquivo`), o link da evidência
       aponta pro arquivo QUE TEM o detalhe, não pro documento atual.
    3. Por fim, cai em `link_arquivo` — o PDF completo do documento atual,
       fallback de último caso.
    """
    if not evidencias:
        return
    if isinstance(evidencias, dict):
        evidencias = [evidencias]
    if not isinstance(evidencias, list):
        return

    query = sb.table("tb_evidencias").select("id_evidencia,descricao,link_evidencia")
    if id_nao_conformidade:
        query = query.eq("id_nao_conformidade", id_nao_conformidade)
    else:
        query = query.eq("id_resposta", id_resposta)
    existentes = {_norm(row.get("descricao")): row for row in query.execute().data}

    for ev in evidencias:
        if isinstance(ev, str):
            ev = {"descricao": ev, "apresentado_por": None}
        if not isinstance(ev, dict):
            continue
        descricao = (ev.get("descricao") or "").strip()
        if not descricao:
            continue

        arquivo_referenciado = _extrai_arquivo_referenciado(descricao)
        link_evidencia = ev.get("_link_pagina")
        if not link_evidencia and arquivo_referenciado:
            link_referenciado = fetch_link_arquivo(sb, arquivo_referenciado)
            if link_referenciado:
                link_evidencia = link_referenciado
        if not link_evidencia:
            link_evidencia = link_arquivo

        existente = existentes.get(_norm(descricao))
        if existente:
            link_atual = existente.get("link_evidencia") or ""
            if link_evidencia and ".html" in link_atual:
                sb.table("tb_evidencias").update(
                    {"link_evidencia": link_evidencia}
                ).eq("id_evidencia", existente["id_evidencia"]).execute()
            continue

        id_evidencia = gen_id("EVID")
        sb.table("tb_evidencias").insert({
            "id_evidencia": id_evidencia,
            "id_nao_conformidade": id_nao_conformidade,
            "id_resposta": id_resposta,
            "apresentado_por": _normaliza_apresentado_por(ev.get("apresentado_por")),
            "descricao": descricao,
            "arquivo_referenciado": arquivo_referenciado,
            "link_evidencia": link_evidencia,
            "sistema_evidencia": ev.get("sistema_evidencia"),
            "equipamento_evidencia": ev.get("equipamento_evidencia"),
            "categoria_evidencia": ev.get("categoria_evidencia"),
            "descritivo_imagens": ev.get("descritivo_imagens"),
            "arquivo_origem": arquivo_origem,
        }).execute()
        existentes[_norm(descricao)] = {"id_evidencia": id_evidencia, "descricao": descricao, "link_evidencia": link_evidencia}


# ---------------------------------------------------------------------------
# Supabase Storage — cada documento sobe o PDF de origem inteiro UMA vez
# (trilha de auditoria, link em `link_arquivo`). O link de evidência
# (`link_evidencia`) reaproveita esse mesmo link, só com um fragmento
# "#page=N" ancorando a página certa (ver `_resolver_link_pagina` em
# streamlit_app.py) — sem gerar nem subir nenhum arquivo extra: aponta pro
# documento original de verdade, com todo o contexto preservado.
# ---------------------------------------------------------------------------
BUCKET_EVIDENCIAS = "evidencias"


def ensure_bucket(sb, nome=BUCKET_EVIDENCIAS):
    """Cria o bucket de evidências no Storage se ainda não existir.

    Idempotente — a API de storage não tem "create if not exists", então só
    ignora o erro de bucket já existente (qualquer outra falha real aparece
    de novo no upload logo em seguida, sem ficar mascarada aqui).
    """
    try:
        sb.storage.create_bucket(nome, options={"public": True})
    except Exception as e:
        if "already exists" not in str(e).lower() and "duplicate" not in str(e).lower():
            raise


def upload_evidencia_arquivo(sb, file_bytes, file_hash, nome_arquivo, bucket=BUCKET_EVIDENCIAS):
    """Sobe o PDF de origem pro Storage e devolve a URL pública.

    Usa o hash do arquivo como nome do objeto — o mesmo PDF nunca sobe duas
    vezes (reprocessamento, ou o mesmo arquivo citado por vários itens só
    reaproveita o link já existente via upsert).
    """
    ensure_bucket(sb, bucket)
    extensao = nome_arquivo.rsplit(".", 1)[-1] if "." in nome_arquivo else "pdf"
    caminho = f"{file_hash}.{extensao}"
    sb.storage.from_(bucket).upload(
        caminho, file_bytes,
        file_options={"content-type": "application/pdf", "upsert": "true"},
    )
    return sb.storage.from_(bucket).get_public_url(caminho)


# ---------------------------------------------------------------------------
# tb_processos_reprocessar — marca um processo inteiro para forçar releitura
# e reescrita mesmo sem o checkbox global de "forçar reprocessamento".
# ---------------------------------------------------------------------------
def marcar_processos_reprocessar(sb, numeros_processo_anp):
    for numero in numeros_processo_anp:
        numero = _norm_processo(numero)
        if numero:
            sb.table("tb_processos_reprocessar").upsert({"numero_processo_anp": numero}).execute()


def fetch_processos_reprocessar(sb):
    res = sb.table("tb_processos_reprocessar").select("*").execute()
    return {_norm_processo(row["numero_processo_anp"]) for row in res.data}


def desmarcar_processo_reprocessar(sb, numero_processo_anp):
    sb.table("tb_processos_reprocessar").delete().eq("numero_processo_anp", _norm_processo(numero_processo_anp)).execute()


# ---------------------------------------------------------------------------
# Leitura para exibição na interface
# ---------------------------------------------------------------------------
def fetch_all(sb, table):
    res = sb.table(table).select("*").order("criado_em", desc=True).execute()
    return res.data

## 5. Inicializar clientes

In [6]:
if not (SUPABASE_URL and SUPABASE_KEY):
    raise ValueError("Faltam credenciais do Supabase (SUPABASE_URL, SUPABASE_KEY) — preencha a seção Configuração acima.")

sb = get_client(SUPABASE_URL, SUPABASE_KEY)

_claude_client = None
if LLM_PROVIDER == "claude":
    if not ANTHROPIC_API_KEY:
        raise ValueError("LLM_PROVIDER está como 'claude', mas falta ANTHROPIC_API_KEY — preencha a seção Configuração acima.")
    from anthropic import Anthropic
    _claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)

if LLM_PROVIDER == "ollama":
    import requests
    try:
        tags = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3).json().get("models", [])
        nomes = [m["name"] for m in tags]
        if not any(OLLAMA_MODEL.split(":")[0] in n for n in nomes):
            print(f"⚠️  Ollama em {OLLAMA_HOST} não tem o modelo '{OLLAMA_MODEL}' baixado ({nomes or 'nenhum'}). Rode: ollama pull {OLLAMA_MODEL}")
    except Exception:
        print(f"⚠️  Não consegui conectar ao Ollama em {OLLAMA_HOST}. Confirme que está rodando e acessível a partir deste notebook.")

print("Clientes inicializados.")


Clientes inicializados.


## 6. Camada de LLM

Wrappers de chamada ao Claude/Ollama e limpeza do JSON de resposta — cópia de
`streamlit_app.py` (seção "Camada de LLM").


In [7]:
import json


def _clean_json_text(raw):
    raw = raw.strip().strip("`")
    if raw.lower().startswith("json"):
        raw = raw[4:].strip()
    # alguns modelos locais "conversam" antes/depois do JSON mesmo quando instruídos a
    # não fazer isso — corta para o primeiro { e o último }
    start, end = raw.find("{"), raw.rfind("}")
    if start != -1 and end != -1:
        raw = raw[start:end + 1]
    return raw


def _call_claude(prompt, max_tokens=8000):
    msg = _claude_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in msg.content if b.type == "text")


def _call_ollama(prompt, num_predict=None):
    import requests
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "think": OLLAMA_THINK,
            "options": {"temperature": 0, "num_ctx": OLLAMA_NUM_CTX, "num_predict": num_predict or OLLAMA_NUM_PREDICT},
        },
        timeout=300,
    )
    resp.raise_for_status()
    data = resp.json()
    texto = data.get("response") or ""
    if not texto.strip():
        pista = " (o modelo pensou mas não respondeu — tente atualizar o Ollama ou trocar de modelo)" if data.get("thinking") else ""
        raise ValueError(f"Ollama devolveu resposta vazia para o modelo '{OLLAMA_MODEL}'{pista}.")
    return texto


def classify_and_extract(texto, tipo_sei=None, nc_existentes=None):
    prompt = build_prompt(texto, tipo_sei=tipo_sei, nao_conformidades_existentes=nc_existentes)
    raw = _call_ollama(prompt) if LLM_PROVIDER == "ollama" else _call_claude(prompt)
    return json.loads(_clean_json_text(raw))


def match_resposta_to_nc(r, ncs):
    """Quando o numero_item extraído não bate com nenhuma NC cadastrada, pergunta à
    IA — com um prompt pequeno, sem reprocessar o PDF inteiro — qual NC real
    corresponde ao assunto já extraído. Retorna o numero_item escolhido, ou None."""
    if not ncs or not (r.get("resumo") or r.get("decisao_anp")):
        return None
    prompt = build_rematch_prompt(r, ncs)
    try:
        raw = _call_ollama(prompt) if LLM_PROVIDER == "ollama" else _call_claude(prompt)
        return json.loads(_clean_json_text(raw)).get("numero_item")
    except Exception:
        return None


## 7. Lógica de vínculo

`apply_auditoria` / `try_apply_respostas` / `retry_pending_queue` — cópia de
`streamlit_app.py`.


In [ ]:
def apply_auditoria(result, filename, is_df=False, link_arquivo=None):
    """auditoria_oficial sempre "resolve" na hora: cria ou atualiza.

    Não conformidades só são gravadas quando "is_df" — o documento é o Documento de
    Fiscalização em si. Ordem de Serviço, Plano e Relatório também chegam aqui como
    auditoria_oficial (enriquecem tb_auditorias), mas nunca devem gerar linha em
    tb_nao_conformidades — essa é exclusivamente do DF (ou de vários DFs, se o
    processo tiver mais de um; upsert_nc já mescla por numero_item).
    """
    aud = {k: v for k, v in {
        "numero_processo_anp": result.get("numero_processo_anp"),
        "numero_relatorio": result.get("numero_relatorio"),
        **(result.get("auditoria") or {}),
    }.items() if v is not None}
    id_auditoria = upsert_auditoria(sb, aud, filename)
    n = 0
    if is_df:
        for nc in result.get("nao_conformidades", []):
            upsert_nc(sb, id_auditoria, nc, filename, link_arquivo=link_arquivo)
        n = len(result.get("nao_conformidades", []))
        # Quando há mais de um DF no mesmo processo, o mais informativo (mais não
        # conformidades + mais campos de auditoria preenchidos) fica como
        # arquivo_origem de referência — não importa a ordem de chegada.
        score = 100 * n + len(aud)
        atualizar_arquivo_origem_df(sb, id_auditoria, filename, score)
    return id_auditoria, n


def try_apply_respostas(numero_processo_anp, numero_relatorio, respostas, tipo, filename, link_arquivo=None):
    """Tenta vincular cada resposta a uma não conformidade existente.

    Retorna (status, itens_nao_resolvidos, mensagem).
    status: 'concluido' | 'parcial' | 'pendente_vinculo'
    """
    aud = find_auditoria(sb, numero_processo_anp, numero_relatorio)
    if not aud:
        return "pendente_vinculo", respostas, "auditoria ainda não cadastrada"

    resolvidos, pendentes = 0, []
    ncs_cache = None
    for r in respostas:
        nc = find_nc(sb, aud["id_auditoria"], r.get("numero_item"))
        if not nc:
            # numero_item não bateu exatamente — o documento pode usar uma numeração
            # própria, sem relação com a numeração das notificações do DF. Pergunta
            # pra IA, só com o resumo já extraído, qual NC real corresponde ao assunto.
            if ncs_cache is None:
                ncs_cache = fetch_ncs_resumo(sb, aud["id_auditoria"])
            numero_recasado = match_resposta_to_nc(r, ncs_cache)
            if numero_recasado:
                nc = find_nc(sb, aud["id_auditoria"], numero_recasado)
        if not nc:
            pendentes.append(r)
            continue
        insert_resposta(sb, nc["id_nao_conformidade"], aud["id_auditoria"], tipo, r, filename, link_arquivo=link_arquivo)
        resolvidos += 1

    if not pendentes:
        return "concluido", [], f"{resolvidos} resposta(s) vinculada(s) à auditoria {aud['id_auditoria']}"
    if resolvidos == 0:
        return "pendente_vinculo", pendentes, "item(ns) de não conformidade ainda não cadastrado(s)"
    return "parcial", pendentes, f"{resolvidos} vinculada(s), {len(pendentes)} aguardando item correspondente"


def retry_pending_queue(on_pendencia=None):
    """Roda depois de toda auditoria nova: tenta resolver a fila de pendências.

    "on_pendencia" (opcional) é chamado a cada item avaliado com
    (nome_arquivo, indice, total) — usado pra UI mostrar progresso enquanto
    a fila inteira é revarrida, já que essa varredura pode envolver uma
    chamada à IA por item (ver `try_apply_respostas` -> `match_resposta_to_nc`).
    """
    resolvidos = []
    pendentes_todos = fetch_pendentes(sb)
    total = len(pendentes_todos)
    for i, item in enumerate(pendentes_todos):
        if on_pendencia:
            on_pendencia(item["nome_arquivo"], i + 1, total)
        payload = item["payload_json"] or {}

        if payload.get("tipo_documento") == "anexo_evidencia":
            motivo = _retry_anexo_pendente(item, payload)
            if motivo:
                resolvidos.append(item["nome_arquivo"])
            continue

        status, pendentes, msg = try_apply_respostas(
            payload.get("numero_processo_anp"),
            payload.get("numero_relatorio"),
            payload.get("respostas", []),
            payload.get("tipo_documento"),
            item["nome_arquivo"],
            link_arquivo=payload.get("link_arquivo"),
        )
        if status != item["status"] or len(pendentes) != len(payload.get("respostas", [])):
            novo_payload = {**payload, "respostas": pendentes} if pendentes else None
            upsert_arquivo_status(
                sb, item["hash_arquivo"], item["nome_arquivo"], item["tipo_documento"],
                item["confianca"], status, msg, novo_payload,
            )
            if status == "concluido":
                resolvidos.append(item["nome_arquivo"])
    return resolvidos


## 8. Processamento de um upload

`finalize_result` é o ponto único onde os três fluxos (upload direto, fila síncrona do
ZIP, aplicação de resultado de lote) convergem para aplicar o gate de confiança, os
overrides do catálogo e gravar no banco — cópia de `streamlit_app.py`.


In [ ]:
def _resolver_link_pagina(pages, trecho, numero_item, link_arquivo):
    """Ancora o link de UMA evidência na página certa do documento ORIGINAL —
    localiza a página pelo `trecho` verbatim (mais preciso), senão pelo `numero_item`,
    e devolve `link_arquivo` com um fragmento "#page=N". Devolve None se não achar."""
    if not link_arquivo:
        return None
    pagina = localizar_pagina_por_trecho(pages, trecho) if trecho else None
    if not pagina and numero_item:
        pagina = localizar_pagina_por_item(pages, numero_item)
    if not pagina:
        return None
    return f"{link_arquivo}#page={pagina}"


def _resolver_paginas_evidencias(pages, link_arquivo, evidencias_por_item):
    """Resolve `ev["_link_pagina"]` in-place pra toda evidência em
    `evidencias_por_item` — lista de {numero_item, evidencias}."""
    for item in evidencias_por_item:
        for ev in item.get("evidencias") or []:
            if not isinstance(ev, dict):
                continue
            link = _resolver_link_pagina(pages, ev.get("trecho"), item.get("numero_item"), link_arquivo)
            if link:
                ev["_link_pagina"] = link


def finalize_result(result, file_hash, nome_arquivo, decisao_catalogo=None, numero_override=None, is_df=False,
                     numero_sei=None, numero_processo_divergente=None, link_arquivo=None, pages=None,
                     skip_retry_queue=False):
    """Aplica overrides determinísticos, o gate de confiança e grava o resultado.

    "skip_retry_queue" evita rodar `retry_pending_queue()` aqui — usado pela
    aplicação de lote da Batch API, que já roda essa fila UMA vez só depois
    de aplicar todos os itens, em vez de uma vez por auditoria concluída
    (ver `process_pending_batches`); upload direto e fila síncrona do ZIP
    continuam resolvendo a fila imediatamente (default `False`).
    """
    tipo = result["tipo_documento"]
    if pages:
        _resolver_paginas_evidencias(
            pages, link_arquivo,
            (result.get("nao_conformidades") or []) + (result.get("respostas") or []),
        )

    # Override determinístico: o catálogo do SEI já classificou este documento como
    # Documento de Fiscalização (ou peça oficial equivalente).
    if decisao_catalogo == "prioritario":
        tipo = result["tipo_documento"] = "auditoria_oficial"

    # Override determinístico: número de processo ANP vem do catálogo CSV.
    if numero_override:
        result["numero_processo_anp"] = numero_override

    rastreio = dict(
        numero_sei=numero_sei,
        numero_processo_anp=result.get("numero_processo_anp"),
        numero_processo_divergente=numero_processo_divergente,
    )

    if decisao_catalogo != "prioritario" and (result["confianca"] < CONFIDENCE_THRESHOLD or tipo in ("indeterminado", "evidencia_anexa")):
        upsert_arquivo_status(sb, file_hash, nome_arquivo, tipo, result["confianca"], "revisao_manual", "confiança baixa ou tipo não processável automaticamente", link_arquivo=link_arquivo, **rastreio)
        return "revisao", f"confiança {result['confianca']:.2f} — tipo {tipo}"

    processo_label = f" [processo {result.get('numero_processo_anp') or '—'}]"

    if tipo == "auditoria_oficial":
        id_auditoria, n = apply_auditoria(result, nome_arquivo, is_df=is_df, link_arquivo=link_arquivo)
        upsert_arquivo_status(sb, file_hash, nome_arquivo, tipo, result["confianca"], "concluido", f"auditoria {id_auditoria}", link_arquivo=link_arquivo, **rastreio)
        detalhe = f"auditoria {id_auditoria} atualizada com {n} não conformidade(s){processo_label}"
        if not skip_retry_queue:
            resolvidos = retry_pending_queue()
            if resolvidos:
                detalhe += f" — também resolveu da fila: {', '.join(resolvidos)}"
        return "concluido", detalhe

    # resposta_operadora ou parecer_anp
    status, pendentes, msg = try_apply_respostas(
        result.get("numero_processo_anp"), result.get("numero_relatorio"),
        result.get("respostas", []), tipo, nome_arquivo, link_arquivo=link_arquivo,
    )
    msg += processo_label
    payload = None
    if status != "concluido":
        payload = {
            "tipo_documento": tipo,
            "numero_processo_anp": result.get("numero_processo_anp"),
            "numero_relatorio": result.get("numero_relatorio"),
            "respostas": pendentes,
            "link_arquivo": link_arquivo,
        }
    upsert_arquivo_status(sb, file_hash, nome_arquivo, tipo, result["confianca"], status, msg, payload, link_arquivo=link_arquivo, **rastreio)
    status_label = {"concluido": "concluido", "parcial": "revisao", "pendente_vinculo": "revisao"}[status]
    return status_label, msg


In [10]:
def _extrair_evidencias_anexo(texto_anexo, ncs_candidatas, id_auditoria, nome_arquivo, link_arquivo, pages=None):
    """Chama a IA UMA VEZ POR NC candidata (nunca a lista inteira numa chamada só —
    resposta grande o bastante estoura max_tokens e volta JSON truncado)."""
    total_evidencias, ncs_atingidas, falhas, erros = 0, 0, 0, []
    for nc_cand in ncs_candidatas:
        try:
            prompt = build_anexo_evidencia_prompt(nc_cand, texto_anexo)
            raw = (
                _call_ollama(prompt, num_predict=ANEXO_OLLAMA_NUM_PREDICT) if LLM_PROVIDER == "ollama"
                else _call_claude(prompt, max_tokens=ANEXO_CLAUDE_MAX_TOKENS)
            )
            evidencias = json.loads(_clean_json_text(raw)).get("evidencias", [])
        except Exception as e:
            falhas += 1
            erros.append(f"{nc_cand.get('numero_item')}: {type(e).__name__}: {e}")
            continue
        if not evidencias:
            continue
        nc = find_nc(sb, id_auditoria, nc_cand.get("numero_item"))
        if not nc:
            continue
        if pages:
            for ev in evidencias:
                if not isinstance(ev, dict):
                    continue
                link = _resolver_link_pagina(pages, ev.get("trecho"), nc_cand.get("numero_item"), link_arquivo)
                if link:
                    ev["_link_pagina"] = link
        insert_evidencias(sb, evidencias, nome_arquivo, id_nao_conformidade=nc["id_nao_conformidade"], link_arquivo=link_arquivo)
        total_evidencias += len(evidencias)
        ncs_atingidas += 1
    return total_evidencias, ncs_atingidas, falhas, erros


def _try_processar_anexo(file_bytes, file_hash, nome_arquivo, doc_id, numero_catalogo):
    """Só chamada para tipo 'Anexo'. Sem processo conhecido, devolve None e o
    chamador descarta normalmente. Com processo conhecido mas sem auditoria/NC
    cadastrada ainda, fica pendente — resolvido depois por retry_pending_queue."""
    if not numero_catalogo:
        return None

    aud = find_auditoria(sb, numero_processo_anp=numero_catalogo)
    ncs = fetch_ncs_resumo(sb, aud["id_auditoria"]) if aud else []

    pages = extract_pages(file_bytes)
    texto_anexo = "\n".join(pages)[:MAX_CHARS]
    if not texto_anexo.strip():
        return None

    link_anexo = upload_evidencia_arquivo(sb, file_bytes, file_hash, nome_arquivo)

    if not ncs:
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, "Anexo", 1.0, "pendente_vinculo",
            "anexo aguardando não conformidades do processo serem cadastradas",
            payload_json={"tipo_documento": "anexo_evidencia", "numero_processo_anp": numero_catalogo},
            numero_sei=doc_id, numero_processo_anp=numero_catalogo,
            link_arquivo=link_anexo,
        )
        return "revisao", "anexo pendente — processo ainda sem não conformidades cadastradas"

    candidatas = filtrar_ncs_relevantes(texto_anexo, ncs)
    if not candidatas:
        motivo = "sem correspondência de vocabulário com as não conformidades do processo — IA não foi chamada"
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, "Anexo", 1.0, "concluido", motivo,
            numero_sei=doc_id, numero_processo_anp=numero_catalogo, link_arquivo=link_anexo,
        )
        return "concluido", motivo

    total_evidencias, ncs_atingidas, falhas, erros = _extrair_evidencias_anexo(
        texto_anexo, candidatas, aud["id_auditoria"], nome_arquivo, link_anexo, pages=pages,
    )

    status = "erro" if falhas and falhas == len(candidatas) else "concluido"
    motivo = (
        f"anexo analisado ({len(candidatas)} NC candidata(s) por vocabulário) — "
        f"{total_evidencias} evidência(s) confirmada(s) para {ncs_atingidas} não conformidade(s)"
        + (f" — {falhas} chamada(s) de IA falharam: {'; '.join(erros)}" if falhas else "")
    )
    upsert_arquivo_status(
        sb, file_hash, nome_arquivo, "Anexo", 1.0 if status == "concluido" else 0.0, status, motivo,
        numero_sei=doc_id, numero_processo_anp=numero_catalogo, link_arquivo=link_anexo,
    )
    return status, motivo


def _retry_anexo_pendente(item, payload):
    """Reaplica _try_processar_anexo pra um Anexo que ficou pendente por falta de NC
    cadastrada, agora que uma auditoria pode ter aparecido. Sem bytes do arquivo em
    memória, baixa de novo do link_arquivo (PDF) já salvo na própria linha."""
    numero_catalogo = payload.get("numero_processo_anp")
    aud = find_auditoria(sb, numero_processo_anp=numero_catalogo)
    if not aud:
        return None
    ncs = fetch_ncs_resumo(sb, aud["id_auditoria"])
    if not ncs:
        return None

    if not item.get("link_arquivo"):
        return None
    try:
        import requests
        resp = requests.get(item["link_arquivo"], timeout=30)
        resp.raise_for_status()
        pages = extract_pages(resp.content)
    except Exception:
        return None
    texto_anexo = "\n".join(pages)[:MAX_CHARS]
    if not texto_anexo.strip():
        return None

    link_anexo = item.get("link_arquivo")
    candidatas = filtrar_ncs_relevantes(texto_anexo, ncs)
    if not candidatas:
        motivo = "sem correspondência de vocabulário com as não conformidades do processo — IA não foi chamada"
    else:
        total_evidencias, ncs_atingidas, falhas, erros = _extrair_evidencias_anexo(
            texto_anexo, candidatas, aud["id_auditoria"], item["nome_arquivo"], link_anexo, pages=pages,
        )
        if falhas and falhas == len(candidatas):
            return None  # todas as chamadas falharam — tenta de novo na próxima passada da fila
        motivo = (
            f"anexo (fila) analisado ({len(candidatas)} NC candidata(s) por vocabulário) — "
            f"{total_evidencias} evidência(s) confirmada(s) para {ncs_atingidas} não conformidade(s)"
            + (f" — {falhas} chamada(s) de IA falharam: {'; '.join(erros)}" if falhas else "")
        )

    upsert_arquivo_status(
        sb, item["hash_arquivo"], item["nome_arquivo"], "Anexo", 1.0, "concluido", motivo,
        numero_sei=item.get("numero_sei"), numero_processo_anp=numero_catalogo,
        link_arquivo=link_anexo,
    )
    return motivo


In [11]:
import hashlib


def process_one_file(file_bytes, nome_arquivo, catalogo=None, force=False, numero_pasta_fallback=None,
                      processos_reprocessar=None):
    """Processa 1 arquivo já em memória (bytes) — chamada de IA síncrona.

    Núcleo compartilhado entre o upload direto e a fila síncrona do processamento em
    lote via ZIP. `numero_pasta_fallback` só é usado pelo caminho do ZIP.

    `processos_reprocessar` é o conjunto (já resolvido antes do laço de arquivos, uma
    vez só) de números de processo marcados para forçar releitura.
    """
    file_hash = hashlib.sha256(file_bytes).hexdigest()

    doc_id = doc_id_from_filename(nome_arquivo)
    entry = (catalogo or {}).get(doc_id)
    decisao_catalogo = classify_catalog_entry(entry)
    numero_catalogo = normalize_processo(entry["processo"]) if entry and entry.get("processo") else None

    if numero_catalogo and processos_reprocessar and numero_catalogo in processos_reprocessar:
        force = True
        desmarcar_processo_reprocessar(sb, numero_catalogo)

    existing = already_processed(sb, file_hash)
    if existing and existing["status"] in ("concluido", "revisao_manual", "erro", "descartado") and not force:
        return "duplicado", f"já processado antes ({existing['status']}) — marque 'forçar reprocessamento' para refazer"

    if decisao_catalogo == "descartar":
        if entry and is_tipo_anexo(entry.get("tipo_documento")):
            resultado_anexo = _try_processar_anexo(file_bytes, file_hash, nome_arquivo, doc_id, numero_catalogo)
            if resultado_anexo:
                return resultado_anexo
        motivo = (
            f"descartado pelo catálogo SEI — tipo '{entry['tipo_documento']}'"
            if entry and entry["status"].strip().lower() == "sucesso"
            else f"descartado pelo catálogo SEI — status de captura '{entry['status']}'" if entry
            else "não mapeado no catálogo CSV"
        )
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, entry.get("tipo_documento") if entry else None, 1.0, "descartado", motivo,
            numero_sei=doc_id, numero_processo_anp=numero_catalogo,
        )
        return "descartado", motivo

    pages = extract_pages(file_bytes)
    texto = "\n".join(pages)[:MAX_CHARS]
    if not texto.strip():
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, None, 0.0, "erro", "sem texto extraível (pode precisar de OCR)",
            numero_sei=doc_id, numero_processo_anp=numero_catalogo,
        )
        return "erro", "sem texto extraível (pode precisar de OCR)"

    tipo_sei = entry["tipo_documento"] if entry else None
    is_df = is_tipo_df(tipo_sei)

    nc_existentes = None
    if not is_df and numero_catalogo:
        aud_existente = find_auditoria(sb, numero_processo_anp=numero_catalogo)
        if aud_existente:
            nc_existentes = fetch_ncs_resumo(sb, aud_existente["id_auditoria"])

    result = classify_and_extract(texto, tipo_sei=tipo_sei, nc_existentes=nc_existentes)

    numero_texto = extract_processo_regex(texto) or numero_pasta_fallback
    numero_divergente = None
    if numero_catalogo:
        numero_override = numero_catalogo
        if numero_texto and normalize_processo(numero_texto) != numero_catalogo:
            numero_divergente = numero_texto
    else:
        numero_override = numero_texto  # sem entrada no catálogo — defensivo

    link_arquivo = upload_evidencia_arquivo(sb, file_bytes, file_hash, nome_arquivo)

    return finalize_result(
        result, file_hash, nome_arquivo, decisao_catalogo, numero_override=numero_override, is_df=is_df,
        numero_sei=doc_id, numero_processo_divergente=numero_divergente, link_arquivo=link_arquivo,
        pages=pages,
    )


## 9. Processamento em lote (ZIP)

`prepare_batch_from_zip` + `submit_batch` (Batch API — só Claude, metade do custo por
token, assíncrono) e `process_zip_batch_sync` (fila síncrona — único caminho para
Ollama, também usável com Claude para resultado imediato). Cópia de `streamlit_app.py`.


In [ ]:
import io as _io
import pathlib
import zipfile


def _iter_zip_pdfs(zip_bytes):
    """Percorre o ZIP inteiro, inclusive subpastas dentro da pasta de cada processo —
    o processo é identificado pelo PRIMEIRO segmento do caminho dentro do zip.

    Devolve os arquivos de cada processo em ordem NUMÉRICA crescente pelo nome
    (nome é o número do documento no SEI, que cresce cronologicamente): processar
    nessa ordem ajuda um documento que cita/depende de outro do MESMO lote resolver
    de primeira, sem precisar da fila de pendências."""
    entradas = []
    with zipfile.ZipFile(_io.BytesIO(zip_bytes)) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            caminho = pathlib.PurePosixPath(info.filename)
            if caminho.suffix.lower() != ".pdf":
                continue
            if any(parte.startswith(".") or parte == "__MACOSX" for parte in caminho.parts):
                continue
            pasta_processo = caminho.parts[0] if len(caminho.parts) > 1 else "sem_pasta"
            entradas.append((pasta_processo, caminho.name, zf.read(info)))
    entradas.sort(key=lambda t: (t[0], chave_ordenacao_arquivo(t[1])))
    yield from entradas


def prepare_batch_from_zip(zip_bytes, catalogo, force=False, processos_reprocessar=None):
    """Varre o ZIP e monta as requisições da Batch API. Descarta, marca erro ou
    duplicado imediatamente (sem gastar tokens) — só os documentos que realmente
    precisam da IA entram no lote. Retorna (lista_de_requests, itens{custom_id:
    detalhes}, contagem)."""
    from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
    from anthropic.types.messages.batch_create_params import Request

    batch_requests = []
    itens = {}
    contagem = {"descartado": 0, "erro": 0, "duplicado": 0, "enviado": 0}

    for i, (pasta_processo, nome_arquivo, file_bytes) in enumerate(_iter_zip_pdfs(zip_bytes)):
        file_hash = hashlib.sha256(file_bytes).hexdigest()

        doc_id = doc_id_from_filename(nome_arquivo)
        entry = catalogo.get(doc_id)
        decisao = classify_catalog_entry(entry)
        numero_catalogo = normalize_processo(entry["processo"]) if entry and entry.get("processo") else None

        force_este = force
        if numero_catalogo and processos_reprocessar and numero_catalogo in processos_reprocessar:
            force_este = True
            desmarcar_processo_reprocessar(sb, numero_catalogo)

        existing = already_processed(sb, file_hash)
        if existing and existing["status"] in ("concluido", "revisao_manual", "erro", "descartado") and not force_este:
            contagem["duplicado"] += 1
            continue

        if decisao == "descartar":
            if entry and is_tipo_anexo(entry.get("tipo_documento")):
                resultado_anexo = _try_processar_anexo(file_bytes, file_hash, nome_arquivo, doc_id, numero_catalogo)
                if resultado_anexo:
                    contagem["anexo_lido"] = contagem.get("anexo_lido", 0) + 1
                    continue
            motivo = (
                f"descartado pelo catálogo SEI — tipo '{entry['tipo_documento']}'"
                if entry and entry["status"].strip().lower() == "sucesso"
                else f"descartado pelo catálogo SEI — status de captura '{entry['status']}'" if entry
                else "não mapeado no catálogo CSV"
            )
            upsert_arquivo_status(
                sb, file_hash, nome_arquivo, entry.get("tipo_documento") if entry else None, 1.0, "descartado", motivo,
                numero_sei=doc_id, numero_processo_anp=numero_catalogo,
            )
            contagem["descartado"] += 1
            continue

        pages = extract_pages(file_bytes)
        texto = "\n".join(pages)[:MAX_CHARS]
        if not texto.strip():
            upsert_arquivo_status(
                sb, file_hash, nome_arquivo, None, 0.0, "erro", "sem texto extraível (pode precisar de OCR)",
                numero_sei=doc_id, numero_processo_anp=numero_catalogo,
            )
            contagem["erro"] += 1
            continue

        tipo_sei = entry["tipo_documento"] if entry else None
        is_df_item = is_tipo_df(tipo_sei)

        nc_existentes = None
        if not is_df_item and numero_catalogo:
            aud_existente = find_auditoria(sb, numero_processo_anp=numero_catalogo)
            if aud_existente:
                nc_existentes = fetch_ncs_resumo(sb, aud_existente["id_auditoria"])

        prompt = build_prompt(texto, tipo_sei=tipo_sei, nao_conformidades_existentes=nc_existentes)

        numero_texto = extract_processo_regex(texto) or extract_processo_from_foldername(pasta_processo)
        numero_divergente = None
        if numero_catalogo:
            numero_final = numero_catalogo
            if numero_texto and normalize_processo(numero_texto) != numero_catalogo:
                numero_divergente = numero_texto
        else:
            numero_final = numero_texto

        link_arquivo = upload_evidencia_arquivo(sb, file_bytes, file_hash, nome_arquivo)

        custom_id = f"item{i:05d}_{file_hash[:8]}"
        batch_requests.append(Request(
            custom_id=custom_id,
            params=MessageCreateParamsNonStreaming(
                model=ANTHROPIC_MODEL,
                max_tokens=8000,
                messages=[{"role": "user", "content": prompt}],
            ),
        ))
        itens[custom_id] = {
            "nome_arquivo": nome_arquivo,
            "hash_arquivo": file_hash,
            "pasta_processo": pasta_processo,
            "tipo_sei": tipo_sei,
            "decisao_catalogo": decisao,
            "is_df": is_df_item,
            "numero_sei": doc_id,
            "numero_processo_final": numero_final,
            "numero_processo_divergente": numero_divergente,
            "link_arquivo": link_arquivo,
        }
        contagem["enviado"] += 1

    return batch_requests, itens, contagem


def process_zip_batch_sync(zip_bytes, catalogo, force=False, on_item=None, processos_reprocessar=None):
    """Processa um ZIP em fila síncrona: chama a IA arquivo por arquivo, aguardando
    cada resposta antes de seguir pro próximo, agrupado por processo. Único caminho
    disponível pra Ollama — mas também serve pra rodar um ZIP com Claude sem usar a
    Batch API, se preferir resultado imediato."""
    itens = list(_iter_zip_pdfs(zip_bytes))
    contagem = {}
    for pasta_processo, nome_arquivo, file_bytes in itens:
        numero_pasta = extract_processo_from_foldername(pasta_processo)
        try:
            status, detalhe = process_one_file(
                file_bytes, nome_arquivo, catalogo=catalogo, force=force,
                numero_pasta_fallback=numero_pasta, processos_reprocessar=processos_reprocessar,
            )
        except Exception as e:
            status, detalhe = "erro", str(e)
        contagem[status] = contagem.get(status, 0) + 1
        if on_item:
            on_item(pasta_processo, nome_arquivo, status, detalhe)
    return contagem


def submit_batch(batch_requests, itens, nome_arquivo_zip=None):
    if not batch_requests:
        return None
    batch = _claude_client.messages.batches.create(requests=batch_requests)
    upsert_lote_batch(sb, batch.id, batch.processing_status, len(itens), itens, nome_arquivo_zip=nome_arquivo_zip)
    return batch.id


def apply_batch_result(item, result, skip_retry_queue=False):
    """Aplica o resultado de UM item de um lote já finalizado (ended)."""
    nome_arquivo = item["nome_arquivo"]
    file_hash = item["hash_arquivo"]
    decisao_catalogo = item["decisao_catalogo"]
    numero_override = item.get("numero_processo_final")

    if result.result.type != "succeeded":
        motivo = f"lote: item {result.result.type}"
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, item.get("tipo_sei"), 0.0, "erro", motivo,
            numero_sei=item.get("numero_sei"), numero_processo_anp=numero_override,
        )
        return "erro", motivo

    raw = "".join(b.text for b in result.result.message.content if b.type == "text")
    try:
        parsed = json.loads(_clean_json_text(raw))
    except json.JSONDecodeError as e:
        motivo = f"lote: resposta da IA não é JSON válido ({e})"
        upsert_arquivo_status(
            sb, file_hash, nome_arquivo, item.get("tipo_sei"), 0.0, "erro", motivo,
            numero_sei=item.get("numero_sei"), numero_processo_anp=numero_override,
        )
        return "erro", motivo

    # Ancorar a página certa por evidência exige o texto por página, que só existe
    # agora — sem mais os bytes do PDF em memória (o lote pode ter levado horas),
    # baixa de novo do link já salvo (best-effort).
    pages = None
    if item.get("link_arquivo"):
        try:
            import requests
            resp = requests.get(item["link_arquivo"], timeout=30)
            resp.raise_for_status()
            pages = extract_pages(resp.content)
        except Exception:
            pages = None

    return finalize_result(
        parsed, file_hash, nome_arquivo, decisao_catalogo, numero_override=numero_override,
        is_df=item.get("is_df", False), numero_sei=item.get("numero_sei"),
        numero_processo_divergente=item.get("numero_processo_divergente"), link_arquivo=item.get("link_arquivo"),
        pages=pages, skip_retry_queue=skip_retry_queue,
    )


def process_pending_batches(on_item=None, on_lote=None, on_pendencia=None):
    """Verifica todos os lotes ainda não concluídos e aplica os resultados prontos.

    Cada item e cada lote são isolados em try/except — um resultado com
    problema (ex: JSON com campo que a IA inventou e não existe na tabela)
    vira só um "erro" registrado naquele arquivo, igual ao upload direto, em
    vez de derrubar a verificação inteira.

    Aplicar um lote grande é lento: cada item leva várias chamadas
    sequenciais ao Supabase. Pra evitar que isso vire trabalho quadrático, a
    fila de pendências NÃO é reprocessada uma vez por auditoria concluída
    (como no upload direto) — cada item aqui é aplicado com
    `skip_retry_queue=True`, e `retry_pending_queue()` roda só UMA vez, no
    final, depois que todos os lotes desta chamada já foram aplicados.

    "on_item" (opcional) é chamado a cada item aplicado com (nome_arquivo,
    status_label, indice, total); "on_lote" (opcional) é chamado nas
    transições de cada lote com (id_lote, indice, total_lotes, fase) — fase
    em "consultando", "pulado", "em_processamento", "aplicando";
    "on_pendencia" (opcional) é repassado pra `retry_pending_queue` no passo
    final.
    """
    resumo = []
    lotes = fetch_lotes_batch(sb, status_excluir="concluido")
    total_lotes = len(lotes)
    algum_item_aplicado = False

    for idx, lote in enumerate(lotes):
        if on_lote:
            on_lote(lote["id_lote"], idx + 1, total_lotes, "consultando")

        if lote["total_itens"] >= BATCH_MAX_ARQUIVOS:
            resumo.append(
                f"{lote['id_lote']}: pulado — lote com {lote['total_itens']} arquivo(s), "
                f"acima do limite de processamento automático ({BATCH_MAX_ARQUIVOS})"
            )
            if on_lote:
                on_lote(lote["id_lote"], idx + 1, total_lotes, "pulado")
            continue
        try:
            batch = _claude_client.messages.batches.retrieve(lote["id_lote"])
        except Exception as e:
            resumo.append(f"{lote['id_lote']}: erro ao consultar o lote — {e}")
            continue

        if batch.processing_status != "ended":
            upsert_lote_batch(sb, lote["id_lote"], batch.processing_status, lote["total_itens"], lote["itens_json"])
            resumo.append(f"{lote['id_lote']}: ainda '{batch.processing_status}'")
            if on_lote:
                on_lote(lote["id_lote"], idx + 1, total_lotes, "em_processamento")
            continue

        itens = lote["itens_json"] or {}
        total = len(itens)
        contagem = {}
        if on_lote:
            on_lote(lote["id_lote"], idx + 1, total_lotes, "aplicando")
        for i, result in enumerate(_claude_client.messages.batches.results(lote["id_lote"])):
            item = itens.get(result.custom_id)
            if not item:
                continue
            try:
                status_label, _ = apply_batch_result(item, result, skip_retry_queue=True)
            except Exception as e:
                upsert_arquivo_status(sb, item["hash_arquivo"], item["nome_arquivo"], item.get("tipo_sei"), 0.0, "erro", f"lote: falha ao aplicar resultado — {e}")
                status_label = "erro"
            contagem[status_label] = contagem.get(status_label, 0) + 1
            algum_item_aplicado = True
            if on_item:
                on_item(item["nome_arquivo"], status_label, i + 1, total)

        upsert_lote_batch(sb, lote["id_lote"], "concluido", lote["total_itens"], itens)
        resumo.append(
            f"{lote['id_lote']}: concluído — {contagem.get('concluido', 0)} ok, "
            f"{contagem.get('revisao', 0)} revisão, {contagem.get('erro', 0)} erro"
        )

    if algum_item_aplicado:
        resolvidos = retry_pending_queue(on_pendencia=on_pendencia)
        if resolvidos:
            resumo.append(f"fila pendente: resolvido(s) {len(resolvidos)} — {', '.join(resolvidos)}")

    return resumo


## 10. Carregar catálogo do SEI

Obrigatório antes de processar qualquer documento — é a fonte de verdade que decide o
que é descartado, o que é Documento de Fiscalização, e qual é o número de processo
autoritativo de cada arquivo.


In [13]:
catalogo = load_catalog(CATALOGO_CSV_PATH)
print(f"Catálogo carregado: {len(catalogo)} documento(s) mapeado(s).")


FileNotFoundError: [Errno 2] No such file or directory: '/lakehouse/default/Files/anp/catalogo.csv'

## 11. Seleção de documentos

Lista os PDFs e ZIPs encontrados no Lakehouse e deixa você escolher, nesta execução,
quais processar — equivalente ao `file_uploader` do app Streamlit.


In [ ]:
def listar_arquivos(pasta, extensoes):
    if not os.path.isdir(pasta):
        return []
    arquivos = []
    for raiz, _, nomes in os.walk(pasta):
        for nome in nomes:
            if os.path.splitext(nome)[1].lower() in extensoes:
                arquivos.append(os.path.join(raiz, nome))
    return sorted(arquivos)


pdfs_disponiveis = listar_arquivos(PDFS_DIR, {".pdf"})
zips_disponiveis = listar_arquivos(LOTES_DIR, {".zip"})
print(f"{len(pdfs_disponiveis)} PDF(s) encontrados em {PDFS_DIR}")
print(f"{len(zips_disponiveis)} ZIP(s) encontrados em {LOTES_DIR}")


In [16]:
import ipywidgets as widgets
from IPython.display import display

sel_pdfs = widgets.SelectMultiple(
    options=pdfs_disponiveis, description="PDFs",
    layout=widgets.Layout(width="100%", height="160px"),
)
sel_zips = widgets.SelectMultiple(
    options=zips_disponiveis, description="ZIPs",
    layout=widgets.Layout(width="100%", height="120px"),
)
chk_force = widgets.Checkbox(value=False, description="Forçar reprocessamento")

display(widgets.VBox([
    widgets.HTML("<b>PDFs para upload direto</b> (Ctrl/Cmd+clique para selecionar vários):"),
    sel_pdfs,
    widgets.HTML("<b>ZIPs para processamento em lote:</b>"),
    sel_zips,
    chk_force,
]))


NameError: name 'pdfs_disponiveis' is not defined

Se os widgets acima não renderizarem de forma interativa nesta sessão do Fabric,
edite `pdfs_selecionados` / `zips_selecionados` / `force_reprocess` diretamente na
célula abaixo (por padrão eles ficam vazios — nada é processado até você escolher).


In [ ]:
pdfs_selecionados = list(sel_pdfs.value)
zips_selecionados = list(sel_zips.value)
force_reprocess = chk_force.value

# Alternativa manual (descomente e ajuste se preferir listar os caminhos direto):
# pdfs_selecionados = [f"{PDFS_DIR}/exemplo1.pdf", f"{PDFS_DIR}/exemplo2.pdf"]
# zips_selecionados = [f"{LOTES_DIR}/lote1.zip"]
# force_reprocess = False

print(f"{len(pdfs_selecionados)} PDF(s) selecionado(s) para upload direto.")
print(f"{len(zips_selecionados)} ZIP(s) selecionado(s) para lote.")
print(f"Forçar reprocessamento: {force_reprocess}")


## 12. (Opcional) marcar processos para forçar reprocessamento

Cola números de processo aqui para forçar a releitura deles na próxima vez que os
arquivos correspondentes passarem por qualquer um dos fluxos abaixo — a marca é
removida automaticamente depois de usada (mesmo comportamento do expander no app).


In [ ]:
numeros_para_marcar = []  # ex: ["48610.221768/2025-20"]

if numeros_para_marcar:
    marcar_processos_reprocessar(sb, numeros_para_marcar)
    print(f"{len(numeros_para_marcar)} processo(s) marcado(s).")

marcados = fetch_processos_reprocessar(sb)
print(f"Atualmente marcado(s): {', '.join(sorted(marcados)) if marcados else '(nenhum)'}")


## 13. Processar PDFs selecionados (upload direto, síncrono)

In [ ]:
resultados_diretos = []

if not catalogo:
    print("❌ Catálogo vazio — carregue o CSV do SEI (seção 10) antes de processar.")
elif not pdfs_selecionados:
    print("Nenhum PDF selecionado (seção 11).")
else:
    processos_reprocessar = fetch_processos_reprocessar(sb)
    icones = {"concluido": "✅", "revisao": "⚠️", "erro": "❌", "duplicado": "↩️", "descartado": "🗑️"}
    for caminho in pdfs_selecionados:
        nome_arquivo = os.path.basename(caminho)
        with open(caminho, "rb") as f:
            file_bytes = f.read()
        try:
            status, detalhe = process_one_file(
                file_bytes, nome_arquivo, catalogo=catalogo, force=force_reprocess,
                processos_reprocessar=processos_reprocessar,
            )
        except Exception as e:
            status, detalhe = "erro", str(e)
        resultados_diretos.append({"arquivo": nome_arquivo, "status": status, "detalhe": detalhe})
        print(f"{icones.get(status, '•')} {nome_arquivo} — {detalhe}")

import pandas as pd
pd.DataFrame(resultados_diretos)


## 14. Processar ZIPs selecionados (lote)

Com `LLM_PROVIDER = "claude"` e `PROCESSAR_ZIP_SINCRONO = False` (padrão), os ZIPs vão
para a **Batch API** da Anthropic (metade do custo por token, resultado em até 24h —
normalmente bem menos; confira o resultado na seção 15). Marque
`PROCESSAR_ZIP_SINCRONO = True` para processar na hora em vez de usar a Batch API
(único modo disponível quando `LLM_PROVIDER = "ollama"`, já que Ollama não tem
equivalente de Batch API).


In [ ]:
PROCESSAR_ZIP_SINCRONO = False

if not catalogo:
    print("❌ Catálogo vazio — carregue o CSV do SEI (seção 10) antes de processar.")
elif not zips_selecionados:
    print("Nenhum ZIP selecionado (seção 11).")
else:
    processos_reprocessar = fetch_processos_reprocessar(sb)

    if LLM_PROVIDER == "claude" and not PROCESSAR_ZIP_SINCRONO:
        for caminho in zips_selecionados:
            nome_zip = os.path.basename(caminho)
            with open(caminho, "rb") as f:
                zip_bytes = f.read()
            print(f"\n{nome_zip}")
            try:
                batch_requests, itens, contagem = prepare_batch_from_zip(
                    zip_bytes, catalogo, force=force_reprocess, processos_reprocessar=processos_reprocessar,
                )
                print(f"  📦 {contagem['enviado']} para a IA · 🗑️ {contagem['descartado']} descartado(s) · "
                      f"📎 {contagem.get('anexo_lido', 0)} anexo(s) lido(s) · ↩️ {contagem['duplicado']} já processado(s) · "
                      f"❌ {contagem['erro']} sem texto extraível")
                if batch_requests:
                    id_lote = submit_batch(batch_requests, itens, nome_arquivo_zip=nome_zip)
                    print(f"  ✅ lote enviado: {id_lote} — volte à seção 15 mais tarde para verificar o resultado.")
                else:
                    print("  ℹ️ nada precisou da IA — nada foi enviado à Batch API.")
            except Exception as e:
                print(f"  ❌ falha ao processar {nome_zip}: {e} — os demais lotes seguem independentes.")
    else:
        icones = {"concluido": "✅", "revisao": "⚠️", "erro": "❌", "duplicado": "↩️", "descartado": "🗑️"}
        for caminho in zips_selecionados:
            nome_zip = os.path.basename(caminho)
            with open(caminho, "rb") as f:
                zip_bytes = f.read()

            def _on_item(pasta_processo, nome_arquivo, status, detalhe, _icones=icones):
                print(f"  {_icones.get(status, '•')} [{pasta_processo}] {nome_arquivo} — {detalhe}")

            print(f"\nProcessando {nome_zip} (fila síncrona)...")
            contagem = process_zip_batch_sync(
                zip_bytes, catalogo, force=force_reprocess, on_item=_on_item, processos_reprocessar=processos_reprocessar,
            )
            print(f"{nome_zip} concluído — {contagem}")


## 15. Verificar / baixar resultados dos lotes pendentes (Batch API)

Rode esta célula sempre que quiser conferir se algum lote enviado na seção 14 já
terminou — pode levar minutos em lotes grandes (cada item aplicado gera algumas
chamadas ao Supabase).


In [ ]:
if LLM_PROVIDER == "claude":
    import time

    inicio = time.time()

    def _decorrido():
        return f"{time.time() - inicio:.0f}s decorridos"

    _fase_label = {
        "consultando": "consultando status na Anthropic",
        "pulado": "pulado — acima do limite de tamanho",
        "em_processamento": "ainda em processamento na Anthropic",
        "aplicando": "aplicando resultados",
    }

    def _on_lote(id_lote, i, total, fase):
        print(f"Lote {i}/{total} ({id_lote}): {_fase_label.get(fase, fase)} — {_decorrido()}")

    def _on_item_lote(nome_arquivo, status_label, i, total):
        print(f"  [{i}/{total}] {nome_arquivo}: {status_label} — {_decorrido()}")

    def _on_pendencia(nome_arquivo, i, total):
        print(f"  Resolvendo fila de pendências... [{i}/{total}] {nome_arquivo} — {_decorrido()}")

    # "consultando"/"em_processamento" podem demorar minutos sem nenhum resultado
    # pronto ainda — sem esses prints intermediários, a célula parece travada.
    resumo = process_pending_batches(on_item=_on_item_lote, on_lote=_on_lote, on_pendencia=_on_pendencia)
    if resumo:
        for linha in resumo:
            print(linha)
    else:
        print("Nenhum lote pendente.")
else:
    print("Ollama não usa Batch API — os ZIPs já foram processados na hora (seção 14).")


## 16. Resolver fila de pendências manualmente

Útil se você processou uma resposta/parecer antes do Documento de Fiscalização
correspondente, ou quer forçar uma nova tentativa sem subir nenhum arquivo novo.


In [ ]:
import time

_inicio_pendentes = time.time()


def _on_pendencia_manual(nome_arquivo, i, total):
    print(f"Verificando pendência... [{i}/{total}] {nome_arquivo} — {time.time() - _inicio_pendentes:.0f}s decorridos")


resolvidos = retry_pending_queue(on_pendencia=_on_pendencia_manual)
print(f"Resolvido(s): {', '.join(resolvidos)}" if resolvidos else "Nada foi resolvido — ainda faltam documentos correspondentes.")


## 17. Dados — visualizar as tabelas gravadas no Supabase

In [17]:
import pandas as pd

df_auditorias = pd.DataFrame(fetch_all(sb, "tb_auditorias"))
display(df_auditorias)


ModuleNotFoundError: No module named 'pandas'

In [ ]:
df_nao_conformidades = pd.DataFrame(fetch_all(sb, "tb_nao_conformidades"))
display(df_nao_conformidades)


In [ ]:
df_respostas = pd.DataFrame(fetch_all(sb, "tb_respostas"))
display(df_respostas)


In [ ]:
df_evidencias = pd.DataFrame(fetch_all(sb, "tb_evidencias"))
display(df_evidencias)
# Cada linha é 1 evidência ligada a UMA não conformidade ou UMA resposta (nunca as
# duas) — "apresentado_por" indica se foi a operadora ou a própria ANP quem apresentou.


In [ ]:
pendentes = [
    a for a in fetch_all(sb, "tb_arquivos_processados")
    if a["status"] in ("pendente_vinculo", "parcial", "revisao_manual", "erro")
]
df_pendentes = pd.DataFrame(pendentes)
display(df_pendentes)
# pendente_vinculo / parcial: aguardando a auditoria ou o item correspondente ser
# cadastrado — rode a seção 16 depois de subir o documento que falta.
# revisao_manual / erro: precisa de conferência humana.


## 18. (Opcional) Preencher retroativamente `norma_referencia` (SGSO) no backlog

Não conformidades gravadas **antes** da extração de `norma_referencia` no formato das Práticas
de Gestão do SGSO (seção 3/7) ficaram com esse campo vazio. Esta célula varre
`tb_nao_conformidades`, pergunta à IA — com um prompt leve, só com o resumo já extraído
(`descricao`, `categoria_nao_conformidade`, `elemento_critico`), sem reabrir nenhum PDF — qual
Prática de Gestão do CATÁLOGO (seção 3) melhor corresponde a cada item, e grava o resultado de
volta no banco.

Rode uma vez só: por padrão só classifica quem está com `norma_referencia` vazio (mude
`SOBRESCREVER_EXISTENTES` para `True` se quiser reclassificar tudo, ex: depois de ajustar o
catálogo de práticas).

In [ ]:
PROMPT_BACKFILL_NORMA = """Você organiza auditorias de segurança operacional da ANP (petróleo e gás, Brasil).

Uma não conformidade já foi extraída de um Documento de Fiscalização, mas o campo
"norma_referencia" (a Prática de Gestão do SGSO violada) não foi preenchido na época.

CATÁLOGO DE PRÁTICAS DE GESTÃO DO SGSO (Regulamento Técnico do Sistema de Gerenciamento da
Segurança Operacional, ANP):
{praticas_gestao_sgso}

Não conformidade:
- descrição: {descricao}
- categoria: {categoria}
- elemento crítico: {elemento_critico}

Qual o número da Prática de Gestão do CATÁLOGO acima que MELHOR corresponde ao ASSUNTO desta não
conformidade? Responda no formato "N" ou "N.M" (N de 1 a 17, M o subitem quando o assunto
corresponder especificamente a um subitem listado) — escolha sempre o número mais próximo, mesmo
por inferência. Responda SOMENTE com JSON, sem markdown, no formato exato
{{"norma_referencia": "string ou null"}} — use null só se o assunto não se encaixar em nenhuma
das 17 práticas.
"""


def build_backfill_norma_prompt(nc):
    return PROMPT_BACKFILL_NORMA.format(
        praticas_gestao_sgso=PRATICAS_GESTAO_SGSO,
        descricao=nc.get("descricao") or "(sem descrição)",
        categoria=nc.get("categoria_nao_conformidade") or "(sem categoria)",
        elemento_critico=nc.get("elemento_critico") or "(não identificado)",
    )


SOBRESCREVER_EXISTENTES = False  # True reclassifica também quem já tem norma_referencia preenchido

todas_ncs = fetch_all(sb, "tb_nao_conformidades")
alvo = [
    nc for nc in todas_ncs
    if SOBRESCREVER_EXISTENTES or not (nc.get("norma_referencia") or "").strip()
]
print(f"{len(alvo)} de {len(todas_ncs)} não conformidade(s) para classificar.")

atualizadas, falhas = 0, []
for i, nc in enumerate(alvo):
    try:
        prompt = build_backfill_norma_prompt(nc)
        raw = _call_ollama(prompt) if LLM_PROVIDER == "ollama" else _call_claude(prompt)
        norma = json.loads(_clean_json_text(raw)).get("norma_referencia")
    except Exception as e:
        falhas.append((nc["id_nao_conformidade"], f"{type(e).__name__}: {e}"))
        print(f"  [{i + 1}/{len(alvo)}] ❌ {nc['id_nao_conformidade']}: {e}")
        continue
    sb.table("tb_nao_conformidades").update(
        {"norma_referencia": norma}
    ).eq("id_nao_conformidade", nc["id_nao_conformidade"]).execute()
    atualizadas += 1
    print(f"  [{i + 1}/{len(alvo)}] ✅ {nc['id_nao_conformidade']} — norma_referencia = {norma}")

print(f"\nConcluído: {atualizadas} atualizada(s), {len(falhas)} falha(s).")
if falhas:
    for id_nc, erro in falhas:
        print(f"  falhou: {id_nc} — {erro}")
